In [ ]:
#%run prelude.rc

import enum
import importlib.util
import sys
from pathlib import Path

import pyarrow


import time

import polars as pl
import numpy as np
import scipy.integrate as integrate
import HErmes as he
import HErmes.fitting as fit
import scipy.stats as st
import matplotlib

from scipy.spatial.transform import Rotation as rot
from datetime import datetime, UTC, timezone
from glob import glob

#pybindings
from pathlib import Path
import dashi as d
d.visual()
import tqdm


import matplotlib.pyplot as plt
import charmingbeauty as cb
lo = cb.layout
cb.visual.set_style_present()



import re
!export DJANGO_ALLOW_ASYNC_UNSAFE=1
import os
from matplotlib import font_manager
from matplotlib import rcParams


os.environ['DJANGO_ALLOW_ASYNC_UNSAFE'] = '1'
plt.rcParams.update({'text.usetex' : False})


from matplotlib import font_manager


rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Open Sans']



import numpy as np
import polars as pl
import numpy as np
import polars as pl

def average_every_n_by_board(df, n, time_col="timestamp", board_col="board_id"):
    if len(df) == 0:
        return df

    if time_col not in df.columns:
        raise ValueError(f'{time_col} not in dataframe')
    if board_col not in df.columns:
        raise ValueError(f'{board_col} not in dataframe')

    out = []
    boards = np.unique(df[board_col].to_numpy())
       
    for b in boards:
        sub = (
            df.filter(pl.col(board_col) == b)
              .sort(time_col)
        )
        
        m = len(sub)
        if m == 0:
            continue

        # make consecutive bins AFTER sorting
        bin_id = np.arange(m) // n
        sub = sub.with_columns(pl.Series("bin_id", bin_id))
        
        exprs = []
        for c, dt in zip(sub.columns, sub.dtypes):
            if c in [board_col, "bin_id"]:
                continue
            if c == time_col:
                exprs.append(pl.col(c).mean().alias(c))
            elif dt.is_numeric():
                exprs.append(pl.col(c).mean().alias(c))
                
        agg = (
            sub.group_by("bin_id", maintain_order=True)
               .agg(exprs)
               .with_columns(pl.lit(b).alias(board_col))
               .drop("bin_id")
               .sort(time_col)
        )
        
        out.append(agg)

    if not out:
        return pl.DataFrame()

    return pl.concat(out).sort([board_col, time_col])





def average_every_n(df, n, time_col="timestamp"):
    if len(df) == 0:
        return df

    if time_col not in df.columns:
        raise ValueError(f'{time_col} not in dataframe')

    sub = df.sort(time_col)

    m = len(sub)
    bin_id = np.arange(m) // n
    sub = sub.with_columns(pl.Series("bin_id", bin_id))

    exprs = []
    for c, dt in zip(sub.columns, sub.dtypes):
        if c == "bin_id":
            continue
        if c == time_col:
            exprs.append(pl.col(c).mean().alias(c))
        elif dt.is_numeric():
            exprs.append(pl.col(c).mean().alias(c))

    return (
        sub.group_by("bin_id", maintain_order=True)
           .agg(exprs)
           .drop("bin_id")
           .sort(time_col)
    )

from pathlib import Path

def compress_df_by_timebin(df, dt=10.0, time_col="timestamp", board_col="board_id"):
    if len(df) == 0:
        return df

    df = df.sort([board_col, time_col])

    df = df.with_columns(
        (pl.col(time_col) / dt).floor().cast(pl.Int64).alias("tbin")
    )

    key_cols = [board_col, "tbin"]

    exprs = [
        pl.col(time_col).mean().alias(time_col)
    ]

    for c, dtp in zip(df.columns, df.dtypes):
        if c in key_cols or c == time_col:
            continue
        if dtp.is_numeric():
            exprs.append(pl.col(c).mean().alias(c))

    out = (
        df.group_by(key_cols)
          .agg(exprs)
          .drop("tbin")
          .sort([board_col, time_col])
    )

    return out




import polars as pl

def shift_local_time(df, last_t, dt, time_col="timestamp"):
    if len(df) == 0:
        return df, last_t

    first_local = float(df[time_col][0])
    offset = last_t + dt - first_local

    df = df.with_columns(
        (pl.col(time_col) + offset).alias(time_col)
    )

    new_last_t = float(df[time_col][-1])
    return df, new_last_t


import numpy as np
import polars as pl

def estimate_dt_by_board(df, board_col="board_id", time_col="timestamp", min_points=5):
    dt_map = {}

    if len(df) == 0:
        return dt_map

    boards = df[board_col].unique().to_list()

    for b in boards:
        sub = (
            df.filter(pl.col(board_col) == b)
              .sort(time_col)
        )

        t = sub[time_col].to_numpy()
        if len(t) < min_points:
            continue

        dt = np.diff(t)
        dt = dt[np.isfinite(dt) & (dt > 0)]

        if len(dt) == 0:
            continue

        dt_map[b] = float(np.median(dt))

    return dt_map

def estimate_dt(df, time_col="timestamp", min_points=5):
    if len(df) == 0:
        return None

    t = df.sort(time_col)[time_col].to_numpy()
    if len(t) < min_points:
        return None

    dt = np.diff(t)
    dt = dt[np.isfinite(dt) & (dt > 0)]

    if len(dt) == 0:
        return None

    return float(np.median(dt))

def shift_local_time_by_board(df, last_t_map, dt_map, board_col="board_id", time_col="timestamp"):
    if len(df) == 0:
        return df, last_t_map

    out = []

    for b in df[board_col].unique().to_list():
        sub = (
            df.filter(pl.col(board_col) == b)
              .sort(time_col)
        )

        if len(sub) == 0:
            continue

        t0 = float(sub[time_col][0])

        last_t = last_t_map.get(b, None)
        dt = dt_map.get(b, None)

        if dt is None:
            # fallback: estimate from this chunk itself
            t = sub[time_col].to_numpy()
            d = np.diff(t)
            d = d[np.isfinite(d) & (d > 0)]
            dt = float(np.median(d)) if len(d) else 0.0

        if last_t is None:
            offset = -t0
        else:
            offset = last_t + dt - t0

        sub = sub.with_columns(
            (pl.col(time_col) + offset).alias(time_col)
        )

        last_t_map[b] = float(sub[time_col].max())
        out.append(sub)

    if not out:
        return df, last_t_map

    return pl.concat(out).sort([board_col, time_col]), last_t_map




def shift_local_time(df, last_t, dt, time_col="timestamp"):
    if len(df) == 0:
        return df, last_t

    df = df.sort(time_col)

    t0 = float(df[time_col][0])

    if dt is None:
        t = df[time_col].to_numpy()
        d = np.diff(t)
        d = d[np.isfinite(d) & (d > 0)]
        dt = float(np.median(d)) if len(d) else 0.0

    if last_t is None:
        offset = -t0
    else:
        offset = last_t + dt - t0

    df = df.with_columns(
        (pl.col(time_col) + offset).alias(time_col)
    )

    last_t = float(df[time_col].max())
    return df, last_t

import polars as pl
import gondola as gon
	
		
		

LTB_to_RB = {
    18: 3,
    2: 32,
    14: 31,
    23: 35,
    3: 23,
    25: 27,
    1: 19,
    4: 16,
    13: 8,
    15: 1,
    5: 26,
    22: 39,
    9: 9,
    7: 41,
    6: 2,
    12: 46,
    21: 7,
    20: 33,
    8: 36,
    11: 28,
}



# all are top rats besides 19; that is switched
PB_to_RB = {
    18: 3,
    2: 32,
    14: 31,
    23: 35,
    3: 23,
    25: 27,
    1: 19,
    4: 16,
    13: 8,
    15: 1,
    5: 26,
    22: 39,
    9: 9,
    7: 41,
    6: 2,
    12: 46,
    21: 7,
    20: 33,
    8: 36,
    11: 28,
}

RB_to_PB = {v: k for k, v in PB_to_RB.items()}

RB_to_LTB = {v: k for k, v in LTB_to_RB.items()}


RAT_to_RB = {
    1:  [3, 15],
    2:  [32, 14],
    3:  [31, 29],
    4:  [35, 13],
    5:  [23, 21],
    6:  [27, 24],
    7:  [20, 19],
    8:  [16, 25],
    9:  [8, 30],
    10: [1, 11],
    11: [26, 22],
    12: [39, 40],
    13: [9, 18],
    14: [41, 42],
    15: [2, 4],
    16: [46, 44],
    17: [7, 17],
    18: [33, 34],
    19: [36, 6],
    20: [28, 5],
}


RB_to_RAT = {}

for rb, rats in RAT_to_RB.items():
    for rat in rats:
        RB_to_RAT[rat] = rb
        

In [ ]:
def build_paddle_map():
    raw = """
1	A   04-11	16
1	B	04-12	16
2	A	04-09	16
2	B	04-10	16
3	A	04-07	16
3	B	04-08	16
4	A	04-05	16
4	B	04-06	16
5	A	04-03	16
5	B	04-04	16
6	A	04-01	16
6	B	04-02	16
7	A	12-16	46
7	B	12-15	46
8	A	12-14	46
8	B	12-13	46
9	A	12-12	46
9	B	12-11	46
10	A	12-10	46
10	B	12-09	46
11	A	12-08	46
11	B	12-07	46
12	A	12-06	46
12	B	12-05	46
13	A	15-02	1
13	B	15-01	1
14	A	15-04	1
14	B	15-03	1
15	A	15-06	1
15	B	15-05	1
16	A	15-08	1
16	B	15-07	1
17	A	15-10	1
17	B	15-09	1
18	A	15-12	1
18	B	15-11	1
19	A	07-05	41
19	B	07-06	41
20	A	07-07	41
20	B	07-08	41
21	A	07-09	41
21	B	07-10	41
22	A	07-11	41
22	B	07-12	41
23	A	07-13	41
23	B	07-14	41
24	A	07-15	41
24	B	07-16	41
25	A	04-14	16
25	B	04-13	16
26	A	04-16	16
26	B	04-15	16
27	A	13-12	8
27	B	13-11	8
28	A	13-10	8
28	B	13-09	8
29	A	13-08	8
29	B	13-07	8
30	A	13-06	8
30	B	13-05	8
31	A	13-04	8
31	B	13-03	8
32	A	13-02	8
32	B	13-01	8
33	A	22-10	39
33	B	22-09	39
34	A	22-08	39
34	B	22-07	39
35	A	22-12	39
35	B	22-11	39
36	A	22-06	39
36	B	22-05	39
37	A	22-14	39
37	B	22-13	39
38	A	22-04	39
38	B	22-03	39
39	A	22-16	39
39	B	22-15	39
40	A	22-02	39
40	B	22-01	39
41	A	12-04	46
41	B	12-03	46
42	A	12-02	46
42	B	12-01	46
43	A	06-06	2
43	B	06-05	2
44	A	06-08	2
44	B	06-07	2
45	A	06-10	2
45	B	06-09	2
46	A	06-12	2
46	B	06-11	2
47	A	06-14	2
47	B	06-13	2
48	A	06-16	2
48	B	06-15	2
49	A	08-10	36
49	B	08-09	36
50	A	08-08	36
50	B	08-07	36
51	A	08-12	36
51	B	08-11	36
52	A	08-06	36
52	B	08-05	36
53	A	08-14	36
53	B	08-13	36
54	A	08-04	36
54	B	08-03	36
55	A	08-16	36
55	B	08-15	36
56	A	08-02	36
56	B	08-01	36
57	A	05-04	26
57	B	05-03	26
58	A	09-14	9
58	B	09-13	9
59	A	21-04	7
59	B	21-03	7
60	A	19-14	20
60	B	19-13	20
61	A	18-11	3
61	B	18-12	3
62	A	18-09	3
62	B	18-10	3
63	A	18-07	3
63	B	18-08	3
64	A	18-05	3
64	B	18-06	3
65	A	18-03	3
65	B	18-04	3
66	A	18-01	3
66	B	18-02	3
67	A	02-02	32
67	B	02-01	32
68	A	02-04	32
68	B	02-03	32
69	A	02-06	32
69	B	02-05	32
70	A	02-08	32
70	B	02-07	32
71	A	02-10	32
71	B	02-09	32
72	A	02-12	32
72	B	02-11	32
73	A	18-13	3
73	B	18-14	3
74	A	18-15	3
74	B	18-16	3
75	A	14-01	31
75	B	14-02	31
76	A	14-03	31
76	B	14-04	31
77	A	14-05	31
77	B	14-06	31
78	A	14-07	31
78	B	14-08	31
79	A	03-15	23
79	B	03-16	23
80	A	03-13	23
80	B	03-14	23
81	A	03-11	23
81	B	03-12	23
82	A	03-09	23
82	B	03-10	23
83	A	03-07	23
83	B	03-08	23
84	A	03-05	23
84	B	03-06	23
85	A	03-03	23
85	B	03-04	23
86	A	03-01	23
86	B	03-02	23
87	A	23-15	35
87	B	23-16	35
88	A	23-13	35
88	B	23-14	35
89	A	23-11	35
89	B	23-12	35
90	A	23-09	35
90	B	23-10	35
91	A	02-14	32
91	B	02-13	32
92	A	02-16	32
92	B	02-15	32
93	A	23-01	35
93	B	23-02	35
94	A	23-03	35
94	B	23-04	35
95	A	23-05	35
95	B	23-06	35
96	A	23-07	35
96	B	23-08	35
97	A	25-15	27
97	B	25-16	27
98	A	25-13	27
98	B	25-14	27
99	A	25-11	27
99	B	25-12	27
100	A	25-09	27
100	B	25-10	27
101	A	25-07	27
101	B	25-08	27
102	A	25-05	27
102	B	25-06	27
103	A	25-03	27
103	B	25-04	27
104	A	25-01	27
104	B	25-02	27
105	A	14-15	31
105	B	14-16	31
106	A	14-13	31
106	B	14-14	31
107	A	14-11	31
107	B	14-12	31
108	A	14-09	31
108	B	14-10	31
109	A	13-16	8
109	B	13-15	8
110	A	13-14	8
110	B	13-13	8
111	A	15-16	1
111	B	15-15	1
112	A	15-14	1
112	B	15-13	1
113	A	05-10	26
113	B	05-09	26
114	A	05-08	26
114	B	05-07	26
115	A	05-06	26
115	B	05-05	26
116	A	19-02	20
116	B	19-01	20
117	A	19-04	20
117	B	19-03	20
118	A	19-06	20
118	B	19-05	20
119	A	11-16	28
119	B	11-15	28
120	A	11-14	28
120	B	11-13	28
121	A	11-12	28
121	B	11-11	28
122	A	11-10	28
122	B	11-09	28
123	A	11-08	28
123	B	11-07	28
124	A	11-06	28
124	B	11-05	28
125	A	11-04	28
125	B	11-03	28
126	A	11-02	28
126	B	11-01	28
127	A	09-16	9
127	B	09-15	9
128	A	05-02	26
128	B	05-01	26
129	A	06-02	2
129	B	06-01	2
130	A	06-04	2
130	B	06-03	2
131	A	07-02	41
131	B	07-01	41
132	A	07-04	41
132	B	07-03	41
133	A	09-08	9
133	B	09-07	9
134	A	09-10	9
134	B	09-09	9
135	A	09-12	9
135	B	09-11	9
136	A	21-16	7
136	B	21-15	7
137	A	21-14	7
137	B	21-13	7
138	A	21-12	7
138	B	21-11	7
139	A	20-02	33
139	B	20-01	33
140	A	20-04	33
140	B	20-03	33
141	A	20-06	33
141	B	20-05	33
142	A	20-08	33
142	B	20-07	33
143	A	20-10	33
143	B	20-09	33
144	A	20-12	33
144	B	20-11	33
145	A	20-14	33
145	B	20-13	33
146	A	20-16	33
146	B	20-15	33
147	A	21-02	7
147	B	21-01	7
148	A	19-16	20
148	B	19-15	20
149	A	05-12	26
149	B	05-11	26
150	A	05-14	26
150	B	05-13	26
151	A	05-16	26
151	B	05-15	26
152	A	09-01	9
152	B	09-02	9
153	A	09-03	9
153	B	09-04	9
154	A	09-05	9
154	B	09-06	9
155	A	21-05	7
155	B	21-06	7
156	A	21-07	7
156	B	21-08	7
157	A	21-09	7
157	B	21-10	7
158	A	19-08	20
158	B	19-07	20
159	A	19-10	20
159	B	19-09	20
160	A	19-12	20
160	B	19-11	20
""".strip().splitlines()

    paddle_map = {}

    for line in raw:
        parts = line.split()
        paddle = int(parts[0])
        side   = parts[1]
        pb_ch  = parts[2]
        rb     = int(parts[3])

        pb, ch = pb_ch.split("-")
        pb = int(pb)
        ch = int(ch)

        signed_id = -paddle if side == "A" else paddle

        paddle_map[signed_id] = {
            "rb": rb,
            "pb": pb,
            "ch": ch
        }

    return paddle_map


paddle_map = build_paddle_map()

def build_pbch_to_paddle_map(paddle_map):
    pbch_to_paddle = {}

    for signed_pid, info in paddle_map.items():
        key = (info["pb"], info["ch"])

        if key in pbch_to_paddle:
            raise ValueError(f"Duplicate mapping for {key}")

        pbch_to_paddle[key] = signed_pid

    return pbch_to_paddle


pbch_to_paddle = build_pbch_to_paddle_map(paddle_map)

# -------------------------------------------------
# paddle_id (signed) -> (ltb_id, ltb_channel)
# A side = negative paddle_id
# B side = positive paddle_id
# -------------------------------------------------

paddle_to_ltb = {}

def add(pid, side, ltb, ch):
    key = -pid if side == "A" else pid
    paddle_to_ltb[key] = (ltb, ch)


# --- fill map ---
add(1,"A",8,11); add(1,"B",8,12)
add(2,"A",8,9);  add(2,"B",8,10)
add(3,"A",8,7);  add(3,"B",8,8)
add(4,"A",8,5);  add(4,"B",8,6)
add(5,"A",8,3);  add(5,"B",8,4)
add(6,"A",8,1);  add(6,"B",8,2)
add(7,"A",16,16); add(7,"B",16,15)
add(8,"A",16,14); add(8,"B",16,13)
add(9,"A",16,12); add(9,"B",16,11)
add(10,"A",16,10); add(10,"B",16,9)
add(11,"A",16,8); add(11,"B",16,7)
add(12,"A",16,6); add(12,"B",16,5)

add(13,"A",10,2); add(13,"B",10,1)
add(14,"A",10,4); add(14,"B",10,3)
add(15,"A",10,6); add(15,"B",10,5)
add(16,"A",10,8); add(16,"B",10,7)
add(17,"A",10,10); add(17,"B",10,9)
add(18,"A",10,12); add(18,"B",10,11)

add(19,"A",14,5); add(19,"B",14,6)
add(20,"A",14,7); add(20,"B",14,8)
add(21,"A",14,9); add(21,"B",14,10)
add(22,"A",14,11); add(22,"B",14,12)
add(23,"A",14,13); add(23,"B",14,14)
add(24,"A",14,15); add(24,"B",14,16)

add(25,"A",8,14); add(25,"B",8,13)
add(26,"A",8,16); add(26,"B",8,15)

add(27,"A",9,12); add(27,"B",9,11)
add(28,"A",9,10); add(28,"B",9,9)
add(29,"A",9,8); add(29,"B",9,7)
add(30,"A",9,6); add(30,"B",9,5)
add(31,"A",9,4); add(31,"B",9,3)
add(32,"A",9,2); add(32,"B",9,1)

add(33,"A",12,10); add(33,"B",12,9)
add(34,"A",12,8); add(34,"B",12,7)
add(35,"A",12,12); add(35,"B",12,11)
add(36,"A",12,6); add(36,"B",12,5)
add(37,"A",12,14); add(37,"B",12,13)
add(38,"A",12,4); add(38,"B",12,3)
add(39,"A",12,16); add(39,"B",12,15)
add(40,"A",12,2); add(40,"B",12,1)

add(41,"A",16,4); add(41,"B",16,3)
add(42,"A",16,2); add(42,"B",16,1)

add(43,"A",15,6); add(43,"B",15,5)
add(44,"A",15,8); add(44,"B",15,7)
add(45,"A",15,10); add(45,"B",15,9)
add(46,"A",15,12); add(46,"B",15,11)
add(47,"A",15,14); add(47,"B",15,13)
add(48,"A",15,16); add(48,"B",15,15)

add(49,"A",19,10); add(49,"B",19,9)
add(50,"A",19,8); add(50,"B",19,7)
add(51,"A",19,12); add(51,"B",19,11)
add(52,"A",19,6); add(52,"B",19,5)
add(53,"A",19,14); add(53,"B",19,13)
add(54,"A",19,4); add(54,"B",19,3)
add(55,"A",19,16); add(55,"B",19,15)
add(56,"A",19,2); add(56,"B",19,1)

add(57,"A",11,4); add(57,"B",11,3)
add(58,"A",13,14); add(58,"B",13,13)
add(59,"A",17,4); add(59,"B",17,3)
add(60,"A",7,14); add(60,"B",7,13)

add(61,"A",1,11); add(61,"B",1,12)
add(62,"A",1,9); add(62,"B",1,10)
add(63,"A",1,7); add(63,"B",1,8)
add(64,"A",1,5); add(64,"B",1,6)
add(65,"A",1,3); add(65,"B",1,4)
add(66,"A",1,1); add(66,"B",1,2)

add(67,"A",2,2); add(67,"B",2,1)
add(68,"A",2,4); add(68,"B",2,3)
add(69,"A",2,6); add(69,"B",2,5)
add(70,"A",2,8); add(70,"B",2,7)
add(71,"A",2,10); add(71,"B",2,9)
add(72,"A",2,12); add(72,"B",2,11)

add(73,"A",1,13); add(73,"B",1,14)
add(74,"A",1,15); add(74,"B",1,16)

add(75,"A",3,1); add(75,"B",3,2)
add(76,"A",3,3); add(76,"B",3,4)
add(77,"A",3,5); add(77,"B",3,6)
add(78,"A",3,7); add(78,"B",3,8)

add(79,"A",5,15); add(79,"B",5,16)
add(80,"A",5,13); add(80,"B",5,14)
add(81,"A",5,11); add(81,"B",5,12)
add(82,"A",5,9); add(82,"B",5,10)
add(83,"A",5,7); add(83,"B",5,8)
add(84,"A",5,5); add(84,"B",5,6)
add(85,"A",5,3); add(85,"B",5,4)
add(86,"A",5,1); add(86,"B",5,2)

add(87,"A",4,15); add(87,"B",4,16)
add(88,"A",4,13); add(88,"B",4,14)
add(89,"A",4,11); add(89,"B",4,12)
add(90,"A",4,9); add(90,"B",4,10)

add(91,"A",2,14); add(91,"B",2,13)
add(92,"A",2,16); add(92,"B",2,15)

add(93,"A",4,1); add(93,"B",4,2)
add(94,"A",4,3); add(94,"B",4,4)
add(95,"A",4,5); add(95,"B",4,6)
add(96,"A",4,7); add(96,"B",4,8)

add(97,"A",6,15); add(97,"B",6,16)
add(98,"A",6,13); add(98,"B",6,14)
add(99,"A",6,11); add(99,"B",6,12)
add(100,"A",6,9); add(100,"B",6,10)
add(101,"A",6,7); add(101,"B",6,8)
add(102,"A",6,5); add(102,"B",6,6)
add(103,"A",6,3); add(103,"B",6,4)
add(104,"A",6,1); add(104,"B",6,2)

add(105,"A",3,15); add(105,"B",3,16)
add(106,"A",3,13); add(106,"B",3,14)
add(107,"A",3,11); add(107,"B",3,12)
add(108,"A",3,9); add(108,"B",3,10)

add(109,"A",9,16); add(109,"B",9,15)
add(110,"A",9,14); add(110,"B",9,13)

add(111,"A",10,16); add(111,"B",10,15)
add(112,"A",10,14); add(112,"B",10,13)

add(113,"A",11,10); add(113,"B",11,9)
add(114,"A",11,8); add(114,"B",11,7)
add(115,"A",11,6); add(115,"B",11,5)

add(116,"A",7,2); add(116,"B",7,1)
add(117,"A",7,4); add(117,"B",7,3)
add(118,"A",7,6); add(118,"B",7,5)

add(119,"A",20,16); add(119,"B",20,15)
add(120,"A",20,14); add(120,"B",20,13)
add(121,"A",20,12); add(121,"B",20,11)
add(122,"A",20,10); add(122,"B",20,9)
add(123,"A",20,8); add(123,"B",20,7)
add(124,"A",20,6); add(124,"B",20,5)
add(125,"A",20,4); add(125,"B",20,3)
add(126,"A",20,2); add(126,"B",20,1)

add(127,"A",13,16); add(127,"B",13,15)
add(128,"A",11,2); add(128,"B",11,1)

add(129,"A",15,2); add(129,"B",15,1)
add(130,"A",15,4); add(130,"B",15,3)

add(131,"A",14,2); add(131,"B",14,1)
add(132,"A",14,4); add(132,"B",14,3)

add(133,"A",13,8); add(133,"B",13,7)
add(134,"A",13,10); add(134,"B",13,9)
add(135,"A",13,12); add(135,"B",13,11)

add(136,"A",17,16); add(136,"B",17,15)
add(137,"A",17,14); add(137,"B",17,13)
add(138,"A",17,12); add(138,"B",17,11)

add(139,"A",18,2); add(139,"B",18,1)
add(140,"A",18,4); add(140,"B",18,3)
add(141,"A",18,6); add(141,"B",18,5)
add(142,"A",18,8); add(142,"B",18,7)
add(143,"A",18,10); add(143,"B",18,9)
add(144,"A",18,12); add(144,"B",18,11)
add(145,"A",18,14); add(145,"B",18,13)
add(146,"A",18,16); add(146,"B",18,15)

add(147,"A",17,2); add(147,"B",17,1)

add(148,"A",7,16); add(148,"B",7,15)

add(149,"A",11,12); add(149,"B",11,11)
add(150,"A",11,14); add(150,"B",11,13)
add(151,"A",11,16); add(151,"B",11,15)

add(152,"A",13,1); add(152,"B",13,2)
add(153,"A",13,3); add(153,"B",13,4)
add(154,"A",13,5); add(154,"B",13,6)

add(155,"A",17,5); add(155,"B",17,6)
add(156,"A",17,7); add(156,"B",17,8)
add(157,"A",17,9); add(157,"B",17,10)

add(158,"A",7,8); add(158,"B",7,7)
add(159,"A",7,10); add(159,"B",7,9)
add(160,"A",7,12); add(160,"B",7,11)

ltb_to_paddles = {}

for pid_signed, (ltb, ch) in paddle_to_ltb.items():
    key = (ltb, ch)
    if key not in ltb_to_paddles:
        ltb_to_paddles[key] = 0

    ltb_to_paddles[key] = pid_signed

In [ ]:
print("gondola version:", gon.__version__)
print("gondola path:", gon.__file__)


db = gon.db


paddles = db.TofPaddle.all()
#print(paddles[1].__dir__())

#(DSI,J, channel) -> (paddle_ID, panel_ID)

dsiJ_chP = db.get_dsi_j_ch_pid_map()
'''for key in dsiJ_chP.keys():
    for key2 in dsiJ_chP[key]:
        for key2 in dsiJ_chP[key]:
            print(key,key2, dsiJ_chP[key][key2])
            print("\n\n")
'''            
pid_dsiJch = {}

for dsi in dsiJ_chP:
    for j in dsiJ_chP[dsi]:
        for ch, (pid, panel) in dsiJ_chP[dsi][j].items():
            key = (pid, panel)
            val = (dsi, j, ch)
            if key not in pid_dsiJch:
                pid_dsiJch[key] = []
            pid_dsiJch[key].append(val)



from collections import defaultdict

# paddles from DB
paddles = db.TofPaddle.all()




# -------------------------------------------------
# (DSI, J) -> list of RB IDs
# because each DSI/J line can have 2 RBs
# -------------------------------------------------
Paddle_to_RAT = defaultdict(set)

dsiJ_to_RBs = defaultdict(set)

for p in paddles:
    key = (int(p.dsi), int(p.j_rb))
    rb_id = int(p.rb_id)
    
    dsiJ_to_RBs[key].add(rb_id)
    Paddle_to_RAT[p.paddle_id].add(RB_to_RAT[p.rb_id])
    

# convert sets to sorted lists
dsiJ_to_RBs = {
    key: sorted(list(rbs))
    for key, rbs in dsiJ_to_RBs.items()
}

print("\n=== (DSI, J) -> RBs ===")
for key in sorted(dsiJ_to_RBs):
    print(f"{key} -> {dsiJ_to_RBs[key]}")


# -------------------------------------------------
# inverse: RB ID -> list of (DSI, J)
# -------------------------------------------------
RB_to_dsiJ = defaultdict(set)

for dsiJ, rbs in dsiJ_to_RBs.items():
    for rb_id in rbs:
        RB_to_dsiJ[rb_id].add(dsiJ)

RB_to_dsiJ = {
    rb_id: sorted(list(dsiJs))
    for rb_id, dsiJs in RB_to_dsiJ.items()
}

print("\n=== RB -> (DSI, J) ===")
for rb_id in sorted(RB_to_dsiJ):
    print(f"RB {rb_id} -> {RB_to_dsiJ[rb_id]}")


In [ ]:
%matplotlib inline


plt.rcParams["font.family"] = "DejaVu Sans"
import gondola as gon
import time

import channel_rates

starttime = 1766959800 #1766115801 # <- this is 10066 #test 1766959800
endtime =   1766969800#1767979800 # <- end   #test 1766979800
'''files = gon.io.grace_get_telemetry_binaries(
    starttime, #start 1765835400
    endtime, #1766949800     1767039800 like 22 hours...
    #, #end time 1767979800 (testing it is for the random 100,000 seconds of flight)
    '/home/gaps/tof-data/antarctica/nextcloud/flight_2025-26'
)'''


chunk_size = 500

# global accumulated outputs
dfPB = dfPA = dfCPU = dfRB = dfLTB = dfMTB = None

# running last times
pa_last_t_map = {}
rb_last_t_map = {}
ltb_last_t_map = {}

cpu_last_t = None
mtb_last_t = None

# cadence maps
pa_dt_map = {}
rb_dt_map = {}
ltb_dt_map = {}

cpu_dt = None
mtb_dt = None


lpt = None
toml_find = False


dfSIP = dfPA = dfCPU = dfRB = dfLTB = dfMTB = None

pa_last_t  = -2.0
cpu_last_t = -2.0
rb_last_t  = -10.0
ltb_last_t = -2.0
mtb_last_t = -10.0
sip_last_t = -30
pb_last_t = -2.0


pa_dt  = 2.0
cpu_dt = 2.0
rb_dt  = 10.0
ltb_dt = 2.0
mtb_dt = 10.0
sip_dt = 30
pb_dt = 2






In [ ]:

#incase we dont want to run again and just load vals in
pa_dt_map = {1.0: 20.0, 2.0: 20.0, 3.0: 20.0, 7.0: 20.0, 9.0: 20.0, 16.0: 20.0, 19.0: 20.0, 23.0: 20.0, 26.0: 20.0, 28.0: 20.0, 31.0: 20.0, 32.0: 20.0, 33.0: 20.0, 35.0: 20.0, 39.0: 20.0, 41.0: 20.0, 46.0: 20.0}
rb_dt_map = {1.0: 10.0, 2.0: 10.0, 3.0: 10.0, 4.0: 10.0, 5.0: 10.0, 6.0: 10.0, 7.0: 10.0, 8.0: 10.0, 9.0: 10.0, 11.0: 10.0, 14.0: 10.0, 15.0: 10.0, 16.0: 10.0, 17.0: 10.0, 18.0: 10.0, 19.0: 10.0, 20.0: 10.0, 21.0: 10.0, 22.0: 10.0, 23.0: 10.0, 24.0: 10.0, 25.0: 10.0, 26.0: 10.0, 28.0: 10.0, 30.0: 10.0, 31.0: 10.0, 32.0: 10.0, 33.0: 10.0, 34.0: 10.0, 35.0: 10.0, 39.0: 10.0, 40.0: 10.0, 41.0: 10.0, 42.0: 10.0, 44.0: 10.0, 46.0: 10.0}
ltb_dt_map = {1.0: 20.0, 2.0: 20.0, 3.0: 20.0, 7.0: 20.0, 8.0: 20.0, 9.0: 20.0, 16.0: 20.0, 19.0: 20.0, 23.0: 20.0, 26.0: 20.0, 28.0: 20.0, 31.0: 20.0, 32.0: 20.0, 33.0: 20.0, 35.0: 20.0, 39.0: 20.0, 41.0: 20.0, 46.0: 20.0}
cpu_dt = 2.0
mtb_dt = 10.0


print("PA dt map:", pa_dt_map)
print("RB dt map:", rb_dt_map)
print("LTB dt map:", ltb_dt_map)
print("CPU dt:", cpu_dt)
print("MTB dt:", mtb_dt)


#sipM.timestamps looks like 30 seconds

plt.rcParams["font.sans-serif"] = ["DejaVu Sans"]  # always available



dfPA  = pl.read_parquet("saved_dfs_absolute_ts/dfPA.parquet")
dfPB  = pl.read_parquet("saved_dfs_absolute_ts/dfPB.parquet")
dfRB  = pl.read_parquet("saved_dfs_absolute_ts/dfRB.parquet")
dfLTB = pl.read_parquet("saved_dfs_absolute_ts/dfLTB.parquet")
dfMTB = pl.read_parquet("saved_dfs_absolute_ts/dfMTB.parquet")
dfCPU = pl.read_parquet("saved_dfs_absolute_ts/dfCPU.parquet")
dfSIP = pl.read_parquet("saved_dfs_absolute_ts/dfSIP.parquet")
dfRates = pl.read_parquet("saved_dfs_absolute_ts/paddle_rates.parquet")

paddleRateTS = dfRates["timestamp"].to_numpy().astype(np.int64)


paddleRates = {
    int(c.replace("paddle_", "")): dfRates[c].to_numpy()
    for c in dfRates.columns if c.startswith("paddle_")
}




dfRatesHit  = pl.read_parquet("saved_dfs_absolute_ts/paddle_rates_hit.parquet")
dfRatesBeta = pl.read_parquet("saved_dfs_absolute_ts/paddle_rates_beta.parquet")

paddleRateTS = dfRatesHit["timestamp"].to_numpy()

paddleRatesHit = {
    int(c.replace("paddle_", "")): dfRatesHit[c].to_numpy()
    for c in dfRatesHit.columns if c.startswith("paddle_")
}

paddleRatesBeta = {
    int(c.replace("paddle_", "")): dfRatesBeta[c].to_numpy()
    for c in dfRatesBeta.columns if c.startswith("paddle_")
}




In [ ]:
pa_temp_cols = [c for c in dfPA.columns if c.startswith("temps")]
pa_bias_cols = [c for c in dfPA.columns if c.startswith("biases")]

rb_temp_cols = [c for c in dfRB.columns if c.startswith("tmp_")]
rb_voltage_cols = [c for c in dfRB.columns if c.endswith("_voltage")]
rb_current_cols = [c for c in dfRB.columns if c.endswith("_current")]
rb_power_cols = [c for c in dfRB.columns if c.endswith("_power")]
rb_env_cols = ["pressure", "humidity"]

cpu_temp_cols = [c for c in dfCPU.columns if "temp" in c.lower()]
cpu_freq_cols = [c for c in dfCPU.columns if "freq" in c.lower()]

ltb_reasonable_cols = ["trenz_temp", "ltb_temp", "thresh0", "thresh1", "thresh2"]

mtb_rate_cols = [
    "trate", "lost_trate", "rb_lost_rate", "tiu_busy_rate",
    "trg_lost_trg_rate", "gaps_blocked_rate", "track_blocked_rate",
    "any_blocked_rate", "trkctrl_blocked_rate"
]



def finite_mask(*arrays):
    mask = np.ones(len(arrays[0]), dtype=bool)
    for a in arrays:
        a = np.asarray(a)
        mask &= np.isfinite(a)
    return mask

def range_mask(x, xmin=None, xmax=None):
    x = np.asarray(x)
    mask = np.isfinite(x)
    if xmin is not None:
        mask &= x >= xmin
    if xmax is not None:
        mask &= x <= xmax
    return mask


# PA
PA_TEMP_MIN, PA_TEMP_MAX = -50, 60
PA_BIAS_MIN, PA_BIAS_MAX = 45, 65

# RB
RB_TEMP_MIN, RB_TEMP_MAX = -50, 90
RB_VOLT_MIN, RB_VOLT_MAX = -5, 10
RB_CURR_MIN, RB_CURR_MAX = -1, 10
RB_PWR_MIN,  RB_PWR_MAX  = -1, 50
HUM_MIN, HUM_MAX = 0, 100
PRESS_MIN, PRESS_MAX = 0, 1200

# CPU
CPU_TEMP_MIN, CPU_TEMP_MAX = -20, 120
CPU_FREQ_MIN, CPU_FREQ_MAX = 0, 5000

# LTB
LTB_TEMP_MIN, LTB_TEMP_MAX = -50, 90
THR_MIN, THR_MAX = 0, 1000

# MTB
RATE_MIN, RATE_MAX = 0, 1e6





In [ ]:
# helper
%matplotlib inline

def get_time(df):
    if "timestamp" in df.columns:
        return df["timestamp"].to_numpy()
    if "total_elapsed" in df.columns:
        return df["total_elapsed"].to_numpy()
    raise KeyError(f"No timestamp-like column found. Columns: {df.columns}")



In [ ]:
def corr_subset(df, cols, cuts=None):
    arrs = []
    names = []
    n = len(df)
    mask = np.ones(n, dtype=bool)
    
    for c in cols:
        x = df[c].to_numpy()
        mask &= np.isfinite(x)
        if cuts and c in cuts:
            lo, hi = cuts[c]
            if lo is not None:
                mask &= x >= lo
            if hi is not None:
                mask &= x <= hi
    
    for c in cols:
        arrs.append(df[c].to_numpy()[mask])
        names.append(c)
        
    A = np.column_stack(arrs)
    C = np.corrcoef(A, rowvar=False)
    return names, C
    
def plot_corr(ax, names, C, title):
    im = ax.imshow(C, vmin=-1, vmax=1)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=90, fontsize=8)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=8)
    ax.set_title(title)
    return im

In [ ]:
plt.rcParams["font.sans-serif"] = ["DejaVu Sans"]  # always available




def nearest_match(x_ref, x_other, max_dt=30):
    x_ref = np.asarray(x_ref)
    x_other = np.asarray(x_other)
    
    if len(x_other) == 0:
        raise ValueError("nearest_match: reference comparison array is empty")
    
    if len(x_other) == 1:
        idx = np.zeros(len(x_ref), dtype=int)
        dt = np.abs(x_other[0] - x_ref)
    else:
        idx = np.searchsorted(x_other, x_ref)
        idx = np.clip(idx, 1, len(x_other) - 1)

        left = idx - 1
        right = idx

        choose_right = np.abs(x_other[right] - x_ref) < np.abs(x_other[left] - x_ref)
        idx = np.where(choose_right, right, left)

        dt = np.abs(x_other[idx] - x_ref)

    # apply max time cut
    if max_dt is not None:
        idx = np.where(dt <= max_dt, idx, -1)

    return idx, dt


def getTempAv(dfPA):
    temp_cols = [c for c in dfPA.columns if c.startswith("temps")]
    
    # stack into matrix: shape (n_rows, 16)
    temps = np.column_stack([dfPA[c].to_numpy() for c in temp_cols])
    # ignore NaNs automatically
    return np.nanmean(temps, axis=1)

    
def getTempAvB(dfPA, boardID):
    temp_cols = [c for c in dfPA.columns if c.startswith("temps")]
    
    sub = (
        dfPA
        .filter(pl.col("board_id") == boardID)
        .group_by("timestamp")
        .agg([
            *[pl.col(c).mean().alias(c) for c in temp_cols]
        ])
        .sort("timestamp")
    )
    
    if len(sub) == 0:
        return np.array([]), np.array([])
    
    temps = np.column_stack([sub[c].to_numpy() for c in temp_cols])
    t = sub["timestamp"].to_numpy()
    avg = np.nanmean(temps, axis=1)
    
    return t, avg


def match_by_board(dfPA, dfRB):
    # pulls arrays to numpy
    pa_board = dfPA["board_id"].to_numpy()
    rb_board = dfRB["board_id"].to_numpy()
    t_pa = dfPA["timestamp"].to_numpy()
    t_rb = dfRB["timestamp"].to_numpy()

    # makes a big array with all the indexs -1 just incase a match is not found, what is a "match"?
    idx_out = np.full(len(dfPA), -1, dtype=int)

    # loop over unique boards present in PA
    for b in np.unique(pa_board):
        #the mask takes a board at a time
        pa_mask = (pa_board == b)
        rb_mask = (rb_board == b)

        #skips all the rb1s
        if np.sum(rb_mask) == 0:
            continue  # no matching RB for this board
        
        t_pa_sub = t_pa[pa_mask]
        t_rb_sub = t_rb[rb_mask]
        
        #takes the timestamp arrays of the same boards
        
        idx_sub, dt = nearest_match(t_pa_sub, t_rb_sub)
        
        # map back to full indices
        rb_indices = np.where(rb_mask)[0]
        idx_out[pa_mask] = rb_indices[idx_sub]
    return idx_out


def getTempByBoardAndChannel(dfPA, boardID, channelID):
    col = f"temps{channelID}"

    if col not in dfPA.columns:
        raise ValueError(f"Column {col} not found in dfPA")

    sub = (
        dfPA
        .filter(pl.col("board_id") == boardID)
        .group_by("timestamp")
        .agg(
            pl.col(col).mean().alias(col)
        )
        .sort("timestamp")
    )

    if len(sub) == 0:
        return np.array([]), np.array([])

    t = sub["timestamp"].to_numpy()
    temp = sub[col].to_numpy()

    return t, temp  

def secToHours(sec):
    return sec/3600

In [ ]:

for name, df in {
    "dfPA": dfPA,
    "dfPB": dfPB,
    "dfRB": dfRB,
    "dfSIP": dfSIP,
    "dfRates": dfRates,
    "dfLTB": dfLTB,
    
    
}.items():
    if df is None or len(df) == 0:
        print(f"\n{name}: empty")
        continue

    ##print(f"\n{name}")
    #print("columns:", df.columns)

    time_col = "timestamp" if "timestamp" in df.columns else "total_elapsed"
    
    t = df[time_col].to_numpy()
    ##print("time_col:", time_col)
    #print("first 10:", t[:10])
    #print("last  10:", t[-10:])
    #print("min/max:", np.nanmin(t), np.nanmax(t))

    if "board_id" in df.columns:
        #print("\nfirst few times by board:")
        for b in np.sort(np.unique(df["board_id"].to_numpy()))[:5]:
            sub = df.filter(pl.col("board_id") == b).sort(time_col)
            #print(f"  board {int(b):2d}:", sub[time_col].to_numpy()[:5])

In [ ]:
%matplotlib inline


def getTempAvB_raw(dfPA, boardID):
    temp_cols = [c for c in dfPA.columns if c.startswith("temps")]

    sub = (
        dfPA
        .filter(pl.col("board_id") == boardID)
        .sort("timestamp")
    )

    if len(sub) == 0:
        return np.array([]), np.array([])

    temps = np.column_stack([sub[c].to_numpy() for c in temp_cols])
    t = sub["timestamp"].to_numpy()
    avg = np.nanmean(temps, axis=1)

    return t, avg

boardID = 7
t, temp_avg = getTempAvB_raw(dfPA, boardID)

plt.figure(figsize=(8,4))
plt.plot(t - t[0], temp_avg, ".")
plt.title(f"Raw average PA temperature, board {boardID}")
plt.xlabel("Time since start (s)")
plt.ylabel("Temp (C)")
plt.show()
    

# temp of PA vs Latitude

In [ ]:
print(dfSIP.columns)

plt.figure(figsize=(10, 5))

legend = []


print(paddles[68])
for I in (range(16)):
    i = I + 1
    t_ex, ex = getTempByBoardAndChannel(dfPA, PB_to_RB[2], i)
    ex = np.asarray(ex)
    ex_sh = ex - np.mean(ex)
    yex = (ex_sh) / np.max(ex_sh)    
    plt.plot(t_ex, yex, "--", alpha=0.2)
    legend.append(f" channel {i}")

    
# --- PA temps ---
for b in np.sort(np.unique(dfPA["board_id"].to_numpy())):
    boardID = int(b)
    if boardID != PB_to_RB[2]:
        continue
    
    t, temp_avg = getTempAvB_raw(dfPA, boardID)

    t = np.asarray(t)
    temp_avg = np.asarray(temp_avg) 

    if len(temp_avg) == 0:
        continue
    temp_avg_sh = temp_avg - np.mean(temp_avg)
    y = (temp_avg_sh) / np.max(temp_avg_sh)

    plt.plot(t, y, ".", alpha=0.7, lw = 0.2)
    legend.append(f"board {boardID} T")


# --- altitude ---
t_alt = np.asarray(dfSIP["timestamp"])
alt = np.asarray(dfSIP["altitude"])

# normalize altitude
alt_sh  = alt - np.mean(alt)
alt_norm = (alt_sh / np.max(alt_sh))

plt.plot(t_alt, alt_norm, ".", alpha=0.9, lw = 3, c = "b")
legend.append("altitude (normalized)")



# --- formatting ---
plt.legend(legend, fontsize=8, ncol=2)
plt.xlabel("Time (s)")
plt.ylabel("Normalized value")
plt.title("PA temperatures and altitude vs time")

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


print("boards")
for b in dfPA["board_id"].unique():
    print(b)


# Rate vs Temp per preamp

In [ ]:

print(dfRB.columns)

import numpy as np

def getTempAv(dfPA):
    temp_cols = [c for c in dfPA.columns if c.startswith("temps")]
    temps = np.column_stack([dfPA[c].to_numpy() for c in temp_cols])
    return np.nanmean(temps, axis=1)

temp_avg = getTempAv(dfPA)

idx = match_by_board(dfPA, dfRB)
valid = idx >= 0

print("len(dfPA)    =", len(dfPA))
print("len(temp_avg)=", len(temp_avg))
print("len(idx)     =", len(idx))
print("len(valid)   =", len(valid))


#print("paddleRateTS range:", paddleRateTS[:3], paddleRateTS[-3:])
#print("t_temp range:", t_temp[:3], t_temp[-3:])


'''
x = temp_avg[valid]
y = dfRB["rate"].to_numpy()[idx[valid]]

plt.figure(figsize=(6,5))
plt.scatter(x, y, s=10, alpha=0.5)

plt.xlabel("Avg PA temp")
plt.ylabel("RB rate")
plt.title("PA temp vs RB rate (same board, time-matched)")

plt.show()

'''


idx = match_by_board(dfPA, dfRB)

valid = idx >= 0  # only rows where match exists

#has the time and the stuff
#paddleRateTS, paddleRates

from scipy.optimize import curve_fit

def line(x, m, b):
    return m * x + b


paddleRateTS = np.asarray(paddleRateTS)
paddleRateTS0 = paddleRateTS# - paddleRateTS[0]

temp_cols = [c for c in dfPA.columns if c.startswith("temps")]
fullRB_to_LTB = {int(p.rb_id): int(p.ltb_id) for p in paddles}

paddle_temp_data = {}

for b in np.unique(dfPA["board_id"].to_numpy()).astype(int):

    if b not in fullRB_to_LTB:
        continue

    ltb_id = fullRB_to_LTB[b]

    for channel in range(1, len(temp_cols) + 1):
        key = (ltb_id, channel)

        if key not in ltb_to_paddles:
            continue

        signed_pid = int(ltb_to_paddles[key])
        paddle_id = abs(signed_pid)

        t_temp, temp = getTempByBoardAndChannel(dfPA, b, channel)
        t_temp = np.asarray(t_temp)
        temp = np.asarray(temp)

        if paddle_id not in paddle_temp_data:
            paddle_temp_data[paddle_id] = []

        paddle_temp_data[paddle_id].append((t_temp, temp))


fit_rows = []
n_plots = 0

for paddle_id in sorted(paddle_temp_data):

    sides = paddle_temp_data[paddle_id]

    if len(sides) < 2:
        print(f"skip paddle {paddle_id}: only found {len(sides)} side(s)")
        continue

    tA, tempA = sides[0]
    tB, tempB = sides[1]

    idx_B, dt_AB = nearest_match(tA, tB, max_dt=30)
    valid_AB = idx_B >= 0

    if np.sum(valid_AB) < 10:
        print(f"skip paddle {paddle_id}: only {np.sum(valid_AB)} A/B temp matches")
        continue

    t_avg = tA[valid_AB]
    temp_avg = 0.5 * (tempA[valid_AB] + tempB[idx_B[valid_AB]])

    if paddle_id in paddleRatesHit:
        rate_key = paddle_id
    elif -paddle_id in paddleRatesHit:
        rate_key = -paddle_id
    else:
        print(f"skip paddle {paddle_id}: no rate found")
        continue

    rate = np.asarray(paddleRatesHit[rate_key])

    idx_temp, dt = nearest_match(paddleRateTS0, t_avg, max_dt=200)
    valid_time = idx_temp >= 0

    if np.sum(valid_time) < 10:
        print(f"skip paddle {paddle_id}: only {np.sum(valid_time)} valid rate/temp matches")
        continue

    x = temp_avg[idx_temp[valid_time]]
    y = rate[valid_time]

    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]
    y = y[m]
    
    if len(x) < 10:
        print(f"skip paddle {paddle_id}: only {len(x)} finite points")
        continue

    try:
        popt, pcov = curve_fit(line, x, y)
        slope, intercept = popt
        slope_err = np.sqrt(pcov[0, 0])

        fit_rows.append({
            "paddle_id": int(paddle_id),
            "slope": float(slope),
            "slope_err": float(slope_err),
            "intercept": float(intercept),
            "n_points": len(x),
        })
        
        
        xfit = np.linspace(np.min(x), np.max(x), 100)
        yfit = line(xfit, slope, intercept)
        
        plt.figure(figsize=(5, 4))
        plt.scatter(x, y, s=10, alpha=0.5)
        plt.plot(xfit, yfit, lw=2, color = "r")

        plt.xlabel("average PA temp A/B (C)")
        plt.ylabel("Paddle rate (Hz)")
        plt.title(
            f"THR0 (Hit) Paddle {paddle_id}: slope = {slope:.3g} ± {slope_err:.3g} Hz/C"
        )

        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        n_plots += 1

    except Exception as e:
        print(f"Fit failed for paddle {paddle_id}: {e}")


print("number of paddle plots made:", n_plots)


if len(fit_rows) > 0:
    pids = np.array([r["paddle_id"] for r in fit_rows])
    slopes = np.array([r["slope"] for r in fit_rows])
    slope_errs = np.array([r["slope_err"] for r in fit_rows])

    order = np.argsort(pids)
    pids = pids[order]
    slopes = slopes[order]
    slope_errs = slope_errs[order]

    groups = [
        (1, 40),
        (41, 80),
        (81, 120),
        (121, 160),
    ]

    for lo, hi in groups:
        m = (pids >= lo) & (pids <= hi)

        if np.sum(m) == 0:
            continue

        plt.figure(figsize=(14, 5))
        plt.errorbar(
            pids[m],
            slopes[m],
            yerr=slope_errs[m],
            fmt="o",
            capsize=3,
            markersize=4,
            linewidth=1,
        )

        plt.axhline(0, lw=1, alpha=0.6)
        plt.xticks(np.arange(lo, hi + 1, 2), rotation=45)
        plt.grid(True, which="both", axis="both", alpha=0.3)

        plt.xlabel("Paddle ID")
        plt.ylabel("Rate-temperature slope (Hz/C)")
        plt.title(f" THR0 (Hit) Rate vs average PA temperature slope by paddle: {lo}–{hi}")

        plt.tight_layout()
        plt.show()
else:
    print("No successful fits.")

#### Beta!!!!! #######

fit_rowsB = []
n_plots = 0

cortina_RATs = {7, 11, 13, 17, 18, 20}

for paddle_id in sorted(paddle_temp_data):

    # --- RAT handling ---
    rat_val = Paddle_to_RAT.get(abs(int(paddle_id)), set())

    if isinstance(rat_val, set):
        rat_set = {int(r) for r in rat_val}
    elif isinstance(rat_val, (list, tuple, np.ndarray)):
        rat_set = {int(r) for r in rat_val}
    elif rat_val is None:
        rat_set = set()
    else:
        rat_set = {int(rat_val)}

    if rat_set & cortina_RATs:
        continue

    rat_label = ",".join(str(r) for r in sorted(rat_set)) if rat_set else "unknown"

    sides = paddle_temp_data[paddle_id]

    if len(sides) < 2:
        continue

    tA, tempA = sides[0]
    tB, tempB = sides[1]

    idx_B, _ = nearest_match(tA, tB, max_dt=30)
    valid_AB = idx_B >= 0

    if np.sum(valid_AB) < 10:
        continue

    t_avg = tA[valid_AB]
    temp_avg = 0.5 * (tempA[valid_AB] + tempB[idx_B[valid_AB]])

    if paddle_id in paddleRatesBeta:
        rate_key = paddle_id
    elif -paddle_id in paddleRatesBeta:
        rate_key = -paddle_id
    else:
        continue

    rate = np.asarray(paddleRatesBeta[rate_key])

    idx_temp, _ = nearest_match(paddleRateTS0, t_avg, max_dt=200)
    valid_time = idx_temp >= 0

    if np.sum(valid_time) < 10:
        continue

    x = temp_avg[idx_temp[valid_time]]
    y = rate[valid_time]

    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]
    y = y[m]

    if len(x) < 10:
        continue

    try:
        popt, pcov = curve_fit(line, x, y)
        slopeB, intercept = popt
        slopeB_err = np.sqrt(pcov[0, 0])

        fit_rowsB.append({
            "paddle_id": int(paddle_id),
            "slopeB": float(slopeB),
            "slopeB_err": float(slopeB_err),
        })

        # --- individual plot ---
        xfit = np.linspace(np.min(x), np.max(x), 100)
        yfit = line(xfit, slopeB, intercept)

        plt.figure(figsize=(5, 4))
        plt.scatter(x, y, s=10, alpha=0.5)
        plt.plot(xfit, yfit, lw=2, color="r")

        plt.xlabel("average PA temp A/B (C)")
        plt.ylabel("Paddle rate (Hz)")
        plt.title(
            f"THR1 (BETA) Paddle {paddle_id}: "
            f"{slopeB:.3g} ± {slopeB_err:.3g} Hz/C"
        )

        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        n_plots += 1

    except Exception as e:
        print(f"Fit failed for paddle {paddle_id}: {e}")

print("number of paddle plots made (BETA):", n_plots)

# -------------------------------------------------
# slope vs paddle ID (BETA)
# -------------------------------------------------
if len(fit_rowsB) > 0:

    pids = np.array([r["paddle_id"] for r in fit_rowsB])
    slopesB = np.array([r["slopeB"] for r in fit_rowsB])
    slope_errsB = np.array([r["slopeB_err"] for r in fit_rowsB])

    order = np.argsort(pids)
    pids = pids[order]
    slopesB = slopesB[order]
    slope_errsB = slope_errsB[order]

    groups = [
        (1, 40),
        (41, 80),
        (81, 120),
        (121, 160),
    ]
    
    for lo, hi in groups:
        m = (pids >= lo) & (pids <= hi)

        if np.sum(m) == 0:
            continue

        plt.figure(figsize=(14, 5))
        plt.errorbar(
            pids[m],
            slopesB[m],
            yerr=slope_errsB[m],
            fmt="o",
            capsize=3,
            markersize=4,
            linewidth=1,
        )
        
        plt.axhline(0, lw=1, alpha=0.6)
        plt.xticks(np.arange(lo, hi + 1, 2), rotation=45)
        plt.grid(True, alpha=0.3)

        plt.xlabel("Paddle ID")
        plt.ylabel("Rate-temperature slope (Hz/C)")
        plt.title(f"THR1 (BETA): slope vs paddle ID ({lo}–{hi})")

        plt.tight_layout()
        plt.show()

else:
    print("No successful BETA fits.")

In [ ]:
# -------------------------------------------------
# slope vs paddle ID (BETA)
# -------------------------------------------------
if len(fit_rowsB) > 0:

    pids = np.array([r["paddle_id"] for r in fit_rowsB])
    slopesB = np.array([r["slopeB"] for r in fit_rowsB])
    slope_errsB = np.array([r["slopeB_err"] for r in fit_rowsB])

    order = np.argsort(pids)
    pids = pids[order]
    slopesB = slopesB[order]
    slope_errsB = slope_errsB[order]
    
    groups = [
        (1, 40),
        (41, 80),
        (81, 120),
        (121, 160),
    ]

    for lo, hi in groups:
        m = (pids >= lo) & (pids <= hi)

        if np.sum(m) == 0:
            continue

        plt.figure(figsize=(14, 5))
        plt.errorbar(
            pids[m],
            slopesB[m],
            yerr=slope_errsB[m],
            fmt="o",
            capsize=3,
            markersize=4,
            linewidth=1,
        )

        plt.axhline(0, lw=1, alpha=0.6)
        plt.xticks(np.arange(lo, hi + 1, 2), rotation=45)
        plt.grid(True, alpha=0.3)

        plt.xlabel("Paddle ID")
        plt.ylabel("Rate-temperature slope (Hz/C)")
        plt.title(f"THR1 (BETA): slope vs paddle ID ({lo}–{hi})")

        plt.tight_layout()
        plt.show()
        
else:
    print("No successful BETA fits.")

# altitude vs rate

In [ ]:
#### HIT (THR0) ALTITUDE ####

fit_rows_alt = []
n_plots = 0

for paddle_id in sorted(paddleRatesHit.keys()):

    if paddle_id < 0:
        continue

    rate = np.asarray(paddleRatesHit[paddle_id])

    if len(rate) != len(paddleRateTS):
        continue

    idx_alt, _ = nearest_match(paddleRateTS, t_alt, max_dt=200)
    valid_time = idx_alt >= 0

    if np.sum(valid_time) < 10:
        continue

    x = alt[idx_alt[valid_time]]
    y = rate[valid_time]

    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]
    y = y[m]

    if len(x) < 10:
        continue

    try:
        popt, pcov = curve_fit(line, x, y)
        slope, intercept = popt
        slope_err = np.sqrt(pcov[0, 0])

        fit_rows_alt.append({
            "paddle_id": int(paddle_id),
            "slope": float(slope),
            "slope_err": float(slope_err),
        })

        xfit = np.linspace(np.min(x), np.max(x), 100)
        yfit = line(xfit, slope, intercept)
        '''
        plt.figure(figsize=(5, 4))
        plt.scatter(x, y, s=10, alpha=0.5)
        plt.plot(xfit, yfit, lw=2, color="r")

        plt.xlabel("Altitude")
        plt.ylabel("Paddle rate (Hz)")
        plt.title(f"THR0 (Hit) Paddle {paddle_id}: {slope:.3g} ± {slope_err:.3g} Hz/alt")

        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()'''

        n_plots += 1

    except Exception as e:
        print(f"Fit failed for paddle {paddle_id}: {e}")

print("HIT altitude plots:", n_plots)




#### BETA (THR1) ALTITUDE ####

fit_rows_altB = []
n_plots = 0

for paddle_id in sorted(paddleRatesBeta.keys()):

    
    # --- RAT handling ---
    rat_val = Paddle_to_RAT.get(abs(int(paddle_id)), set())

    if isinstance(rat_val, set):
        rat_set = {int(r) for r in rat_val}
    elif isinstance(rat_val, (list, tuple, np.ndarray)):
        rat_set = {int(r) for r in rat_val}
    elif rat_val is None:
        rat_set = set()
    else:
        rat_set = {int(rat_val)}

    if rat_set & cortina_RATs:
        continue
        
    if paddle_id < 0:
        continue

    rate = np.asarray(paddleRatesBeta[paddle_id])

    if len(rate) != len(paddleRateTS):
        continue

    idx_alt, _ = nearest_match(paddleRateTS, t_alt, max_dt=200)
    valid_time = idx_alt >= 0

    if np.sum(valid_time) < 10:
        continue

    x = alt[idx_alt[valid_time]]
    y = rate[valid_time]

    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]
    y = y[m]

    if len(x) < 10:
        continue

    try:
        popt, pcov = curve_fit(line, x, y)
        slopeB, intercept = popt
        slopeB_err = np.sqrt(pcov[0, 0])

        fit_rows_altB.append({
            "paddle_id": int(paddle_id),
            "slopeB": float(slopeB),
            "slopeB_err": float(slopeB_err),
        })

        xfit = np.linspace(np.min(x), np.max(x), 100)
        yfit = line(xfit, slopeB, intercept)
        '''
        plt.figure(figsize=(5, 4))
        plt.scatter(x, y, s=10, alpha=0.5)
        plt.plot(xfit, yfit, lw=2, color="r")

        plt.xlabel("Altitude")
        plt.ylabel("Paddle rate (Hz)")
        plt.title(f"THR1 (BETA) Paddle {paddle_id}: {slopeB:.3g} ± {slopeB_err:.3g} Hz/alt")

        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()'''

        n_plots += 1

    except Exception as e:
        print(f"Fit failed for paddle {paddle_id}: {e}")

print("BETA altitude plots:", n_plots)






if len(fit_rows_alt) > 0:

    pids = np.array([r["paddle_id"] for r in fit_rows_alt])
    slopes = np.array([r["slope"] for r in fit_rows_alt])
    slope_errs = np.array([r["slope_err"] for r in fit_rows_alt])

    order = np.argsort(pids)
    pids, slopes, slope_errs = pids[order], slopes[order], slope_errs[order]

    for lo, hi in [(1,40),(41,80),(81,120),(121,160)]:

        m = (pids >= lo) & (pids <= hi)
        if np.sum(m) == 0:
            continue
        
        plt.figure(figsize=(14,5))
        plt.errorbar(pids[m], slopes[m], yerr=slope_errs[m], fmt="o", capsize=3)

        plt.axhline(0, lw=1)
        plt.xticks(np.arange(lo, hi+1, 2), rotation=45)
        plt.grid(True, alpha=0.3)

        plt.xlabel("Paddle ID")
        plt.ylabel("Rate-altitude slope")
        plt.title(f"THR0 (Hit) altitude slope ({lo}–{hi})")

        plt.tight_layout()
        plt.show()



if len(fit_rows_altB) > 0:

    pids = np.array([r["paddle_id"] for r in fit_rows_altB])
    slopesB = np.array([r["slopeB"] for r in fit_rows_altB])
    slope_errsB = np.array([r["slopeB_err"] for r in fit_rows_altB])

    order = np.argsort(pids)
    pids, slopesB, slope_errsB = pids[order], slopesB[order], slope_errsB[order]

    for lo, hi in [(1,40),(41,80),(81,120),(121,160)]:

        m = (pids >= lo) & (pids <= hi)
        if np.sum(m) == 0:
            continue

        plt.figure(figsize=(14,5))
        plt.errorbar(pids[m], slopesB[m], yerr=slope_errsB[m], fmt="o", capsize=3)

        plt.axhline(0, lw=1)
        plt.xticks(np.arange(lo, hi+1, 2), rotation=45)
        plt.grid(True, alpha=0.3)

        plt.xlabel("Paddle ID")
        plt.ylabel("Rate-altitude slope")
        plt.title(f"THR1 (BETA) altitude slope ({lo}–{hi})")

        plt.tight_layout()
        plt.show()
        
        

In [ ]:
boardID = 7

sub = (
    dfPA
    .filter(pl.col("board_id") == boardID)
    .sort("timestamp")
)

print(
    sub.group_by("timestamp")
       .len()
       .sort("timestamp")
       .head(30)
)
print(dfPA)

In [ ]:


temp_cols = [c for c in dfPA.columns if c.startswith("temps")]
boards = np.unique(dfPA["board_id"].to_numpy())

for b in boards:
    mask = (dfPA["board_id"].to_numpy() == b)

    if np.sum(mask) < 5:
        continue

    t = dfPA["timestamp"].to_numpy()[mask]

    temps = np.column_stack([
        dfPA[c].to_numpy()[mask] for c in temp_cols
    ])

    avg = np.nanmean(temps, axis=1)
    std = np.nanstd(temps, axis=1)
    '''
    plt.figure(figsize=(8,5))

    # individual channels
    for i in range(temps.shape[1]):
        plt.scatter(t, temps[:, i], alpha=0.3)

    # average
    plt.plot(t, avg, linewidth=2, label="avg")

    # spread band
    plt.fill_between(t, avg-std, avg+std, alpha=0.2)

    plt.xlabel("Time")
    plt.ylabel("Temp (C)")
    plt.title(f"Board {b} temps (with avg + spread)")
    plt.legend()
    plt.show()'''

    

# Alt bins, temp range

In [ ]:
import numpy as np
import matplotlib.pyplot as plt



roi_low_percentile = 5
roi_high_percentile = 95

# -------------------------------------------------
# SIP altitude setup: reference timestamps
# -------------------------------------------------
t_alt = np.asarray(dfSIP["timestamp"])
alt = np.asarray(dfSIP["altitude"])

m_alt = np.isfinite(t_alt) & np.isfinite(alt)
t_alt = t_alt[m_alt]
alt = alt[m_alt]

order_alt = np.argsort(t_alt)
t_alt = t_alt[order_alt]
alt = alt[order_alt]

# -------------------------------------------------
# build board-temperature matrix on SIP timestamps
# rows = SIP timestamps
# cols = boards
# -------------------------------------------------
boards = np.sort(np.unique(dfPA["board_id"].to_numpy()))

temp_matrix = np.full((len(t_alt), len(boards)), np.nan)

for j, b in enumerate(boards):
    boardID = int(b)

    t_pa, temp_avg = getTempAvB_raw(dfPA, boardID)

    t_pa = np.asarray(t_pa)
    temp_avg = np.asarray(temp_avg)

    m = np.isfinite(t_pa) & np.isfinite(temp_avg)
    t_pa = t_pa[m]
    temp_avg = temp_avg[m]

    if len(t_pa) == 0:
        continue

    # make sure PA temps are sorted
    order = np.argsort(t_pa)
    t_pa = t_pa[order]
    temp_avg = temp_avg[order]

    # for every SIP timestamp, find nearest PA temp timestamp
    idx, dt = nearest_match(t_alt, t_pa, max_dt=30)

    valid = idx >= 0
    temp_matrix[valid, j] = temp_avg[idx[valid]]

# -------------------------------------------------
# average across boards at each timestamp
# -------------------------------------------------
temp_avg_all_boards = np.nanmean(temp_matrix, axis=1)
n_boards_used = np.sum(np.isfinite(temp_matrix), axis=1)

valid_avg = np.isfinite(temp_avg_all_boards) & np.isfinite(alt)

t_common = t_alt[valid_avg]
alt_common = alt[valid_avg]
temp_common = temp_avg_all_boards[valid_avg]
n_boards_common = n_boards_used[valid_avg]

print("number of common averaged points:", len(temp_common))
print("median boards contributing:", np.nanmedian(n_boards_common))




# -------------------------------------------------
# define ROI via percentile bands in altitude bins
# -------------------------------------------------
n_bins = 40  # finer than before for smooth envelope

bins = np.linspace(np.nanmin(alt_common), np.nanmax(alt_common), n_bins + 1)
bin_centers = 0.5 * (bins[:-1] + bins[1:])

p_low = np.full(n_bins, np.nan)
p_high = np.full(n_bins, np.nan)
p_med = np.full(n_bins, np.nan)

for i in range(n_bins):
    if i == n_bins - 1:
        m = (alt_common >= bins[i]) & (alt_common <= bins[i+1])
    else:
        m = (alt_common >= bins[i]) & (alt_common < bins[i+1])

    temps = temp_common[m]

    if len(temps) < 10:  # avoid garbage bins
        continue

    p_low[i]  = np.percentile(temps, roi_low_percentile)   # lower bound
    p_high[i] = np.percentile(temps, roi_high_percentile)   # upper bound
    p_med[i]  = np.percentile(temps, 50)   # median



# -------------------------------------------------
# plot scatter + ROI band
# -------------------------------------------------
plt.figure(figsize=(10, 5))

# original scatter (light)
plt.plot(alt_common, temp_common, ".", alpha=0.15, label="data")

# ROI shaded region
plt.fill_between(
    bin_centers,
    p_low,
    p_high,
    alpha=0.3,
    color = 'orange',
    label="5–95% band"
)


plt.xlabel("Altitude")
plt.ylabel("Average PA temperature")
plt.title("Board-averaged PA temperature vs altitude (ROI overlay)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.ylim(-15,8)
plt.show()






# -------------------------------------------------
# bin by altitude and plot ROI temp range as bars
# -------------------------------------------------
n_bins = 20

bins = np.linspace(np.nanmin(alt_common), np.nanmax(alt_common), n_bins + 1)
bin_centers = 0.5 * (bins[:-1] + bins[1:])
bin_widths = np.diff(bins)

temp_min = np.full(n_bins, np.nan)
temp_max = np.full(n_bins, np.nan)
temp_mean = np.full(n_bins, np.nan)
temp_std = np.full(n_bins, np.nan)
n_in_bin = np.zeros(n_bins, dtype=int)
n_roi_in_bin = np.zeros(n_bins, dtype=int)

for i in range(n_bins):
    if i == n_bins - 1:
        m_bin = (alt_common >= bins[i]) & (alt_common <= bins[i + 1])
    else:
        m_bin = (alt_common >= bins[i]) & (alt_common < bins[i + 1])

    temps_here = temp_common[m_bin]
    temps_here = temps_here[np.isfinite(temps_here)]

    n_in_bin[i] = len(temps_here)

    if len(temps_here) == 0:
        continue

    # -----------------------------
    # define ROI inside this alt bin
    # -----------------------------
    t_low = np.percentile(temps_here, roi_low_percentile)
    t_high = np.percentile(temps_here, roi_high_percentile)

    temps_roi = temps_here[
        (temps_here >= t_low) &
        (temps_here <= t_high)
    ]

    n_roi_in_bin[i] = len(temps_roi)

    if len(temps_roi) == 0:
        continue

    temp_min[i] = np.nanmin(temps_roi)
    temp_max[i] = np.nanmax(temps_roi)
    temp_mean[i] = np.nanmean(temps_roi)
    temp_std[i] = np.nanstd(temps_roi)


# -------------------------------------------------
# convert min/max into bar bottoms and heights
# -------------------------------------------------
bar_bottom = temp_min
bar_height = temp_max - temp_min

valid = (
    np.isfinite(bin_centers) &
    np.isfinite(bar_bottom) &
    np.isfinite(bar_height)
)

plt.figure(figsize=(9, 5))

plt.bar(
    bin_centers[valid],
    bar_height[valid],
    width=bin_widths[valid],
    bottom=bar_bottom[valid],
    alpha=0.35,
    align="center",
    edgecolor="k",
    label=f"{roi_low_percentile}–{roi_high_percentile}% temp ROI"
)

# mean marker inside each bar
plt.plot(
    bin_centers[valid],
    temp_mean[valid],
    "ko",
    ms=4,
    label="ROI mean"
)

plt.xlabel("Altitude [m]")
plt.ylabel("Temperature across all TOF Pre-Amps [C]")
plt.title(
    f"ROI range of board-averaged PA temperature in altitude bins\n"
    f"{n_bins} bins, bin width ≈ {np.nanmean(bin_widths):.1f} m"
)

plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()






# Temp bins Alt range

In [ ]:
# -------------------------------------------------
# ROI band: altitude vs temperature (flipped axes)
# -------------------------------------------------
n_bins = 20

roi_low_percentile = 5
roi_high_percentile = 95

# bin in temperature
bins = np.linspace(np.nanmin(temp_common), np.nanmax(temp_common), n_bins + 1)
bin_centers = 0.5 * (bins[:-1] + bins[1:])

alt_low = np.full(n_bins, np.nan)
alt_high = np.full(n_bins, np.nan)
alt_med = np.full(n_bins, np.nan)

for i in range(n_bins):
    if i == n_bins - 1:
        m = (temp_common >= bins[i]) & (temp_common <= bins[i+1])
    else:
        m = (temp_common >= bins[i]) & (temp_common < bins[i+1])

    alt_here = alt_common[m]
    alt_here = alt_here[np.isfinite(alt_here)]

    if len(alt_here) < 10:
        continue

    alt_low[i]  = np.percentile(alt_here, roi_low_percentile)
    alt_high[i] = np.percentile(alt_here, roi_high_percentile)
    alt_med[i]  = np.percentile(alt_here, 50)


# -------------------------------------------------
# plot
# -------------------------------------------------
plt.figure(figsize=(10, 5))

# light scatter (optional but helpful)
plt.plot(temp_common, alt_common, ".", alpha=0.1, label="data")

# ROI band (now vertical in sense of original)
plt.fill_between(
    bin_centers,
    alt_low,
    alt_high,
    alpha=0.3,
    color="orange",
    label=f"{roi_low_percentile}–{roi_high_percentile}% altitude band"
)

# median line
plt.plot(bin_centers, alt_med, "r-", lw=2, label="median altitude")
plt.xlim(-15,8)

plt.xlabel("Average temperature across all TOF Pre-Amps [C]")
plt.ylabel("Altitude [m]")
plt.title("Altitude vs temperature (ROI band)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()






# -------------------------------------------------
# bin by temperature and plot ROI altitude range as vertical bars
# -------------------------------------------------
n_bins = 20

bins = np.linspace(np.nanmin(temp_common), np.nanmax(temp_common), n_bins + 1)
bin_centers = 0.5 * (bins[:-1] + bins[1:])
bin_widths = np.diff(bins)

alt_min = np.full(n_bins, np.nan)
alt_max = np.full(n_bins, np.nan)
alt_mean = np.full(n_bins, np.nan)

n_in_bin = np.zeros(n_bins, dtype=int)
n_roi_in_bin = np.zeros(n_bins, dtype=int)

for i in range(n_bins):
    if i == n_bins - 1:
        m_bin = (temp_common >= bins[i]) & (temp_common <= bins[i + 1])
    else:
        m_bin = (temp_common >= bins[i]) & (temp_common < bins[i + 1])

    alt_here = alt_common[m_bin]
    alt_here = alt_here[np.isfinite(alt_here)]

    n_in_bin[i] = len(alt_here)

    if len(alt_here) == 0:
        continue

    a_low = np.percentile(alt_here, roi_low_percentile)
    a_high = np.percentile(alt_here, roi_high_percentile)

    alt_roi = alt_here[
        (alt_here >= a_low) &
        (alt_here <= a_high)
    ]

    n_roi_in_bin[i] = len(alt_roi)

    if len(alt_roi) == 0:
        continue

    alt_min[i] = np.nanmin(alt_roi)
    alt_max[i] = np.nanmax(alt_roi)
    alt_mean[i] = np.nanmean(alt_roi)


# -------------------------------------------------
# vertical bars: x = temp, y = altitude range
# -------------------------------------------------
bar_bottom = alt_min
bar_height = alt_max - alt_min

valid = (
    np.isfinite(bin_centers) &
    np.isfinite(bar_bottom) &
    np.isfinite(bar_height)
)

plt.figure(figsize=(9, 5))

plt.bar(
    bin_centers[valid],
    bar_height[valid],
    width=bin_widths[valid],
    bottom=bar_bottom[valid],
    alpha=0.35,
    align="center",
    edgecolor="k",
    label=f"{roi_low_percentile}–{roi_high_percentile}% altitude ROI"
)

plt.plot(
    bin_centers[valid],
    alt_mean[valid],
    "ko",
    ms=4,
    label="ROI mean altitude"
)

plt.xlabel("Average temperature across all TOF Pre-Amps [C]")
plt.ylabel("Altitude [m]")
plt.title(
    f"ROI altitude range in temperature bins\n"
    f"{n_bins} bins, temp bin width ≈ {np.nanmean(bin_widths):.2f} C"
)
plt.xlim(-15,8)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()




# Mean alt in $\pm$ 100 m; temp anywhere. 

In [ ]:
#### HIT/BETA RATE VS TEMP AT MEAN ALTITUDE ####

from scipy.optimize import curve_fit
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd





# -------------------------------------------------
# optional global time selection
# -------------------------------------------------
use_time_window = True

t_start = 1.7665e9  # set to None if unused
t_stop  = 1.768e9

def apply_time_cut(t, *arrays):
    if not use_time_window or t_start is None or t_stop is None:
        return (t, *arrays)

    m = (t >= t_start) & (t <= t_stop)

    out = [t[m]]
    for arr in arrays:
        out.append(arr[m])

    return tuple(out)





def line(x, m, b):
    return m * x + b

threshold_name = "BETA"   # :HIT" or "BETA"

fit_rows_temp_at_alt = []
n_plots = 0

umb_cubetop_panels = [1, 7, 8, 9, 10, 11, 12, 13]

alt_window = 100
half_window = alt_window / 2
max_dt = 30

# -------------------------------------------------
# altitude reference
# -------------------------------------------------
t_alt = np.asarray(dfSIP["timestamp"])
alt = np.asarray(dfSIP["altitude"])

m_alt = np.isfinite(t_alt) & np.isfinite(alt)
t_alt = t_alt[m_alt]
alt = alt[m_alt]
t_alt, alt = apply_time_cut(t_alt, alt)


order_alt = np.argsort(t_alt)
t_alt = t_alt[order_alt]
alt = alt[order_alt]
    
# -------------------------------------------------
# loop over paddles
# -------------------------------------------------
for p in paddles:
    paddle_id = int(p.paddle_id)
    
    if p.panel_id not in umb_cubetop_panels:
        continue
    # -------------------------------------------------
    # rate setup
    # -------------------------------------------------
    if threshold_name == "HIT":
        rate_full = np.asarray(paddleRatesHit[paddle_id])
    elif threshold_name == "BETA":
        rate_full = np.asarray(paddleRatesBeta[paddle_id])
    else:
        raise ValueError("threshold_name must be 'HIT' or 'BETA'")
        
    m_rate = np.isfinite(t_rate) & np.isfinite(rate)
    t_rate = t_rate[m_rate]
    rate = rate[m_rate]
    
    t_rate, rate = apply_time_cut(t_rate, rate)
    print(paddle_id)
    # make sure sorted
    order_rate = np.argsort(t_rate)
    t_rate = t_rate[order_rate]
    rate = rate[order_rate]
    
    if len(rate) < 10:
        continue


    # -------------------------------------------------
    # temperature reference for each paddle
    # average the two preamp channels corresponding to this paddle
    # -------------------------------------------------
    
    chA = int(p.ltb_chA)
    chB = int(p.ltb_chB)
    b = paddle_map[paddle_id]['rb']
    
    tA, tempA = getTempByBoardAndChannel(dfPA, b, chA)
    tB, tempB = getTempByBoardAndChannel(dfPA, b, chB)

    tA = np.asarray(tA)
    tempA = np.asarray(tempA)
    tB = np.asarray(tB)
    tempB = np.asarray(tempB)
    
    mA = np.isfinite(tA) & np.isfinite(tempA)
    mB = np.isfinite(tB) & np.isfinite(tempB)
    
    tA = tA[mA]
    tempA = tempA[mA]
    tB = tB[mB]
    tempB = tempB[mB]
    
    tA, tempA = apply_time_cut(tA, tempA)
    tB, tempB = apply_time_cut(tB, tempB)
    
    # sort both channels
    orderA = np.argsort(tA)
    tA = tA[orderA]
    tempA = tempA[orderA]

    orderB = np.argsort(tB)
    tB = tB[orderB]
    tempB = tempB[orderB]
    if len(tA) < 10 or len(tB) < 10:
        continue
    # match B to A timestamps
    idxB, dtB = nearest_match(tA, tB, max_dt=max_dt)
    
    valid_temp = idxB >= 0
    
    if np.sum(valid_temp) < 10:
        continue
    t_temp = tA[valid_temp]
    temp = 0.5 * (tempA[valid_temp] + tempB[idxB[valid_temp]])
    
    # final clean/sort
    m_temp = np.isfinite(t_temp) & np.isfinite(temp)
    t_temp = t_temp[m_temp]
    temp = temp[m_temp]
    
    order_temp = np.argsort(t_temp)
    t_temp = t_temp[order_temp]
    temp = temp[order_temp]
    
    # -------------------------------------------------
    # match rate timestamps to altitude and temperature
    # -------------------------------------------------
    idx_alt, dt_alt = nearest_match(t_rate, t_alt, max_dt=max_dt)
    idx_temp, dt_temp = nearest_match(t_rate, t_temp, max_dt=max_dt)

    valid = (idx_alt >= 0) & (idx_temp >= 0)

    if np.sum(valid) < 10:
        continue

    t_rate = t_rate[valid]
    rate = rate[valid]
    alt_match = alt[idx_alt[valid]]
    temp_match = temp[idx_temp[valid]]

    # -------------------------------------------------
    # choose altitude ROI around mean altitude
    # -------------------------------------------------
    alt_mean = np.nanmean(alt_match)

    m_alt_roi = (
        np.isfinite(alt_match) &
        np.isfinite(temp_match) &
        np.isfinite(rate) &
        (alt_match >= alt_mean - half_window) &
        (alt_match <= alt_mean + half_window)
    )

    rate_roi = rate[m_alt_roi]
    temp_roi = temp_match[m_alt_roi]
    alt_roi = alt_match[m_alt_roi]

    if len(rate_roi) < 10:
        continue

    # -------------------------------------------------
    # fit rate vs temperature inside altitude ROI
    # -------------------------------------------------
    try:
        
        popt, pcov = curve_fit(line, temp_roi, rate_roi)
        m_fit, b_fit = popt
        m_err, b_err = np.sqrt(np.diag(pcov))
    except Exception as e:
        print(f"fit failed for paddle {paddle_id}: {e}")
        continue

    fit_rows_temp_at_alt.append({
        "paddle_id": paddle_id,
        "threshold": threshold_name,
        "alt_mean": alt_mean,
        "alt_low": alt_mean - half_window,
        "alt_high": alt_mean + half_window,
        "n_points": len(rate_roi),
        "temp_min": np.nanmin(temp_roi),
        "temp_max": np.nanmax(temp_roi),
        "rate_vs_temp_slope": m_fit,
        "rate_vs_temp_slope_err": m_err,
        "intercept": b_fit,
        "intercept_err": b_err,
    })

    # -------------------------------------------------
    # plot rate vs temp at roughly fixed altitude
    # -------------------------------------------------
    xfit = np.linspace(np.nanmin(temp_roi), np.nanmax(temp_roi), 200)

    plt.figure(figsize=(7, 5))

    plt.plot(
        temp_roi,
        rate_roi,
        ".",
        alpha=0.35,
        label=f"{threshold_name} data"
    )

    plt.plot(
        xfit,
        line(xfit, m_fit, b_fit),
        "r-",
        lw=2,
        label=f"slope = {m_fit:.3e} ± {m_err:.3e} rate/C"
    )

    plt.xlabel("Average PA temperature for given Pre-Amp [C]")
    plt.ylabel(f"{threshold_name} rate")
    plt.title(
        f"Paddle {paddle_id}: {threshold_name} rate vs temperature\n"
        f"Altitude ROI: {alt_mean - half_window:.0f}–{alt_mean + half_window:.0f} m"
    )

    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    n_plots += 1


fit_df_temp_at_alt = pd.DataFrame(fit_rows_temp_at_alt)
fit_df_temp_at_alt




import numpy as np
import matplotlib.pyplot as plt

df = fit_df_temp_at_alt.copy()

# split
df_low = df[df["paddle_id"] < 13].sort_values("paddle_id")
df_high = df[df["paddle_id"] >= 13].sort_values("paddle_id")


def make_plot(df_sub, title_suffix):

    paddles = df_sub["paddle_id"].to_numpy()
    slopes = df_sub["rate_vs_temp_slope"].to_numpy()
    errors = df_sub["rate_vs_temp_slope_err"].to_numpy()
    npts = df_sub["n_points"].to_numpy()

    plt.figure(figsize=(7, 5))

    plt.errorbar(
        paddles,
        slopes,
        yerr=errors,
        fmt="o",
        capsize=4
    )

    # annotate n points
    for x, y, n in zip(paddles, slopes, npts):
        plt.text(x, y, f"{n}", fontsize=8, ha="center", va="bottom")

    # weighted mean
    weights = 1 / errors**2
    wmean = np.sum(weights * slopes) / np.sum(weights)
    werr = np.sqrt(1 / np.sum(weights))

    plt.axhline(0, color="k", ls="--", alpha=0.5)
    plt.axhline(wmean, color="r", lw=2, label=f"mean = {wmean:.3f} ± {werr:.3f}")

    plt.xlabel("Paddle ID")
    plt.ylabel("Slope [rate / °C]")
    plt.title(
        f"{threshold_name} slope per paddle ({title_suffix})\n"
        f"Altitude slice: {df_sub['alt_low'].iloc[0]:.0f}–{df_sub['alt_high'].iloc[0]:.0f} m"
    )

    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


# make both plots
make_plot(df_low, "paddle_id < 13")
make_plot(df_high, "paddle_id ≥ 13")

# rate vs altitude at fixed temp window mean +/- 1 C 


In [ ]:
#### HIT/BETA RATE VS ALTITUDE AT FIXED TEMP WINDOW ####



# -------------------------------------------------
# optional global time selection
# -------------------------------------------------
use_time_window = True

t_start = 1.7665e9  # set to None if unused
t_stop  = 1.768e9

def apply_time_cut(t, *arrays):
    if not use_time_window or t_start is None or t_stop is None:
        return (t, *arrays)

    m = (t >= t_start) & (t <= t_stop)

    out = [t[m]]
    for arr in arrays:
        out.append(arr[m])

    return tuple(out)


    




from scipy.optimize import curve_fit
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

def line(x, m, b):
    return m * x + b

threshold_name = "BETA"   # "HIT" or "BETA"

fit_rows_alt_at_temp = []
n_plots = 0

umb_cubetop_panels = [1, 7, 8, 9, 10, 11, 12, 13]

temp_window = 1.0      # total window in C
half_temp_window = temp_window / 2
max_dt = 30

# -------------------------------------------------
# altitude reference
# -------------------------------------------------
t_alt = np.asarray(dfSIP["timestamp"])
alt = np.asarray(dfSIP["altitude"])

m_alt = np.isfinite(t_alt) & np.isfinite(alt)
t_alt = t_alt[m_alt]
alt = alt[m_alt]

#timestamp cut
t_alt, alt = apply_time_cut(t_alt, alt)

order_alt = np.argsort(t_alt)
t_alt = t_alt[order_alt]
alt = alt[order_alt]

# -------------------------------------------------
# temperature reference
# board-averaged temp time series
# -------------------------------------------------
t_temp = np.asarray(t_common)
temp = np.asarray(temp_common)

m_temp = np.isfinite(t_temp) & np.isfinite(temp)
t_temp = t_temp[m_temp]
temp = temp[m_temp]

#timestamp cut
t_temp, temp = apply_time_cut(t_temp, temp)

order_temp = np.argsort(t_temp)
t_temp = t_temp[order_temp]
temp = temp[order_temp]


# -------------------------------------------------
# loop over paddles
# -------------------------------------------------
for p in paddles:

    paddle_id = int(p.paddle_id)

    if p.panel_id not in umb_cubetop_panels:
        continue

    if threshold_name == "HIT":
        rate = np.asarray(paddleRatesHit[paddle_id])
    elif threshold_name == "BETA":
        rate = np.asarray(paddleRatesBeta[paddle_id])
    else:
        raise ValueError("threshold_name must be 'HIT' or 'BETA'")

    t_rate = np.asarray(paddleRateTS)

    m_rate = np.isfinite(t_rate) & np.isfinite(rate)
    t_rate = t_rate[m_rate]
    rate = rate[m_rate]

    #timestamp cut
    t_rate, rate = apply_time_cut(t_rate, rate)
    if len(rate) < 10:
        continue

    # -------------------------------------------------
    # match rate timestamps to altitude and temperature
    # -------------------------------------------------
    idx_alt, dt_alt = nearest_match(t_rate, t_alt, max_dt=max_dt)
    idx_temp, dt_temp = nearest_match(t_rate, t_temp, max_dt=max_dt)

    valid = (idx_alt >= 0) & (idx_temp >= 0)

    if np.sum(valid) < 10:
        continue

    t_rate = t_rate[valid]
    rate = rate[valid]
    alt_match = alt[idx_alt[valid]]
    temp_match = temp[idx_temp[valid]]

    # -------------------------------------------------
    # paddle-specific temperature ROI
    # -------------------------------------------------
    temp_mean = np.nanmean(temp_match)

    m_temp_roi = (
        np.isfinite(alt_match) &
        np.isfinite(temp_match) &
        np.isfinite(rate) &
        (temp_match >= temp_mean - half_temp_window) &
        (temp_match <= temp_mean + half_temp_window)
    )

    rate_roi = rate[m_temp_roi]
    alt_roi = alt_match[m_temp_roi]
    temp_roi = temp_match[m_temp_roi]

    if len(rate_roi) < 10:
        continue

    # -------------------------------------------------
    # fit rate vs altitude inside temp ROI
    # -------------------------------------------------
    try:
        popt, pcov = curve_fit(line, alt_roi, rate_roi)
        m_fit, b_fit = popt
        m_err, b_err = np.sqrt(np.diag(pcov))
    except Exception as e:
        print(f"fit failed for paddle {paddle_id}: {e}")
        continue

    fit_rows_alt_at_temp.append({
        "paddle_id": paddle_id,
        "threshold": threshold_name,
        "temp_mean": temp_mean,
        "temp_low": temp_mean - half_temp_window,
        "temp_high": temp_mean + half_temp_window,
        "n_points": len(rate_roi),
        "alt_min": np.nanmin(alt_roi),
        "alt_max": np.nanmax(alt_roi),
        "rate_vs_alt_slope": m_fit,
        "rate_vs_alt_slope_err": m_err,
        "intercept": b_fit,
        "intercept_err": b_err,
    })

    # -------------------------------------------------
    # plot rate vs altitude at roughly fixed temp
    # -------------------------------------------------
    xfit = np.linspace(np.nanmin(alt_roi), np.nanmax(alt_roi), 200)

    plt.figure(figsize=(7, 5))

    plt.plot(
        alt_roi,
        rate_roi,
        ".",
        alpha=0.35,
        label=f"{threshold_name} data"
    )

    plt.plot(
        xfit,
        line(xfit, m_fit, b_fit),
        "r-",
        lw=2,
        label=f"slope = {m_fit:.3e} ± {m_err:.3e} rate/m"
    )

    plt.xlabel("Altitude [m]")
    plt.ylabel(f"{threshold_name} rate")
    plt.title(
        f"Paddle {paddle_id}: {threshold_name} rate vs altitude\n"
        f"Temp ROI: {temp_mean - half_temp_window:.2f}–{temp_mean + half_temp_window:.2f} C"
    )

    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    n_plots += 1


fit_df_alt_at_temp = pd.DataFrame(fit_rows_alt_at_temp)
fit_df_alt_at_temp



df = fit_df_alt_at_temp.copy()

df_low = df[df["paddle_id"] < 13].sort_values("paddle_id")
df_high = df[df["paddle_id"] >= 13].sort_values("paddle_id")

def make_plot_alt(df_sub, title_suffix):

    paddles = df_sub["paddle_id"].to_numpy()
    slopes = df_sub["rate_vs_alt_slope"].to_numpy()
    errors = df_sub["rate_vs_alt_slope_err"].to_numpy()
    npts = df_sub["n_points"].to_numpy()

    plt.figure(figsize=(7, 5))

    plt.errorbar(
        paddles,
        slopes,
        yerr=errors,
        fmt="o",
        capsize=4
    )

    for x, y, n in zip(paddles, slopes, npts):
        plt.text(x, y, f"{n}", fontsize=8, ha="center", va="bottom")

    weights = 1 / errors**2
    wmean = np.sum(weights * slopes) / np.sum(weights)
    werr = np.sqrt(1 / np.sum(weights))

    plt.axhline(0, color="k", ls="--", alpha=0.5)
    plt.axhline(wmean, color="r", lw=2, label=f"mean = {wmean:.3e} ± {werr:.3e}")

    plt.xlabel("Paddle ID")
    plt.ylabel("Slope [rate / m]")
    plt.title(
        f"{threshold_name} rate vs altitude slope ({title_suffix})\n"
        f"Temp window: mean ± {half_temp_window:.2f} C"
    )

    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


make_plot_alt(df_low, "paddle_id < 13")
make_plot_alt(df_high, "paddle_id ≥ 13")



In [ ]:
#### HIT/BETA RATE VS ALTITUDE WITH EACH PA TEMP HELD NEAR ITS OWN MEAN ####

from scipy.optimize import curve_fit
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

def line(x, m, b):
    return m * x + b

# -------------------------------------------------
# optional global time selection
# -------------------------------------------------
use_time_window = True

t_start = 1.7665e9
t_stop  = 1.768e9

def apply_time_cut(t, *arrays):
    t = np.asarray(t)

    if not use_time_window or t_start is None or t_stop is None:
        return (t, *arrays)

    m = (t >= t_start) & (t <= t_stop)

    out = [t[m]]
    for arr in arrays:
        arr = np.asarray(arr)
        out.append(arr[m])

    return tuple(out)


threshold_name = "BETA"   # "HIT" or "BETA"

fit_rows_alt_at_pa_temp = []
n_plots = 0

umb_cubetop_panels = [1, 7, 8, 9, 10, 11, 12, 13]

temp_half_window = 1   # each PA mean ± 1 C
max_dt = 30              # seconds


# -------------------------------------------------
# altitude reference
# -------------------------------------------------
t_alt = np.asarray(dfSIP["timestamp"])
alt = np.asarray(dfSIP["altitude"])

m_alt = np.isfinite(t_alt) & np.isfinite(alt)
t_alt = t_alt[m_alt]
alt = alt[m_alt]

t_alt, alt = apply_time_cut(t_alt, alt)

order_alt = np.argsort(t_alt)
t_alt = t_alt[order_alt]
alt = alt[order_alt]


# -------------------------------------------------
# loop over paddles
# -------------------------------------------------
for p in paddles:

    paddle_id = int(p.paddle_id)

    if p.panel_id not in umb_cubetop_panels:
        continue

    # -------------------------------------------------
    # rate setup
    # -------------------------------------------------
    if threshold_name == "HIT":
        rate_full = np.asarray(paddleRatesHit[paddle_id])
    elif threshold_name == "BETA":
        rate_full = np.asarray(paddleRatesBeta[paddle_id])
    else:
        raise ValueError("threshold_name must be 'HIT' or 'BETA'")

    t_rate_full = np.asarray(paddleRateTS)

    # force rate/time same length
    n = min(len(t_rate_full), len(rate_full))
    t_rate = t_rate_full[:n]
    rate = rate_full[:n]

    m_rate = np.isfinite(t_rate) & np.isfinite(rate)
    t_rate = t_rate[m_rate]
    rate = rate[m_rate]

    t_rate, rate = apply_time_cut(t_rate, rate)

    order_rate = np.argsort(t_rate)
    t_rate = t_rate[order_rate]
    rate = rate[order_rate]

    if len(rate) < 10:
        continue

    # -------------------------------------------------
    # PA temperature setup for this paddle
    # -------------------------------------------------
    chA = int(p.ltb_chA)
    chB = int(p.ltb_chB)

    b = paddle_map[paddle_id]["rb"]

    tA, tempA = getTempByBoardAndChannel(dfPA, b, chA)
    tB, tempB = getTempByBoardAndChannel(dfPA, b, chB)

    tA = np.asarray(tA)
    tempA = np.asarray(tempA)
    tB = np.asarray(tB)
    tempB = np.asarray(tempB)

    mA = np.isfinite(tA) & np.isfinite(tempA)
    mB = np.isfinite(tB) & np.isfinite(tempB)

    tA = tA[mA]
    tempA = tempA[mA]
    tB = tB[mB]
    tempB = tempB[mB]

    tA, tempA = apply_time_cut(tA, tempA)
    tB, tempB = apply_time_cut(tB, tempB)

    orderA = np.argsort(tA)
    tA = tA[orderA]
    tempA = tempA[orderA]

    orderB = np.argsort(tB)
    tB = tB[orderB]
    tempB = tempB[orderB]

    if len(tA) < 10 or len(tB) < 10:
        continue

    # -------------------------------------------------
    # match rate timestamps to altitude, PA A, and PA B
    # -------------------------------------------------
    idx_alt, dt_alt = nearest_match(t_rate, t_alt, max_dt=max_dt)
    idx_A, dt_A = nearest_match(t_rate, tA, max_dt=max_dt)
    idx_B, dt_B = nearest_match(t_rate, tB, max_dt=max_dt)

    valid = (idx_alt >= 0) & (idx_A >= 0) & (idx_B >= 0)

    if np.sum(valid) < 10:
        continue

    t_rate_match = t_rate[valid]
    rate_match = rate[valid]

    alt_match = alt[idx_alt[valid]]
    tempA_match = tempA[idx_A[valid]]
    tempB_match = tempB[idx_B[valid]]

    # -------------------------------------------------
    # each PA gets its own mean and own ±0.5 C window
    # -------------------------------------------------
    tempA_mean = np.nanmean(tempA_match)
    tempB_mean = np.nanmean(tempB_match)

    m_temp_roi = (
        np.isfinite(alt_match) &
        np.isfinite(rate_match) &
        np.isfinite(tempA_match) &
        np.isfinite(tempB_match) &

        (tempA_match >= tempA_mean - temp_half_window) &
        (tempA_match <= tempA_mean + temp_half_window) &

        (tempB_match >= tempB_mean - temp_half_window) &
        (tempB_match <= tempB_mean + temp_half_window)
    )

    rate_roi = rate_match[m_temp_roi]
    alt_roi = alt_match[m_temp_roi]
    tempA_roi = tempA_match[m_temp_roi]
    tempB_roi = tempB_match[m_temp_roi]

    if len(rate_roi) < 10:
        continue

    # useful display temp only, not used for cut
    temp_pair_avg_roi = 0.5 * (tempA_roi + tempB_roi)

    # -------------------------------------------------
    # fit rate vs altitude with PA temps constrained
    # -------------------------------------------------
    try:
        popt, pcov = curve_fit(line, alt_roi, rate_roi)
        m_fit, b_fit = popt
        m_err, b_err = np.sqrt(np.diag(pcov))
    except Exception as e:
        print(f"fit failed for paddle {paddle_id}: {e}")
        continue

    fit_rows_alt_at_pa_temp.append({
        "paddle_id": paddle_id,
        "panel_id": int(p.panel_id),
        "threshold": threshold_name,
        "rb_id": b,
        "chA": chA,
        "chB": chB,

        "tempA_mean": tempA_mean,
        "tempA_low": tempA_mean - temp_half_window,
        "tempA_high": tempA_mean + temp_half_window,

        "tempB_mean": tempB_mean,
        "tempB_low": tempB_mean - temp_half_window,
        "tempB_high": tempB_mean + temp_half_window,

        "n_points": len(rate_roi),
        "alt_min": np.nanmin(alt_roi),
        "alt_max": np.nanmax(alt_roi),
        "temp_pair_avg_min": np.nanmin(temp_pair_avg_roi),
        "temp_pair_avg_max": np.nanmax(temp_pair_avg_roi),

        "rate_vs_alt_slope": m_fit,
        "rate_vs_alt_slope_err": m_err,
        "intercept": b_fit,
        "intercept_err": b_err,
    })

    # -------------------------------------------------
    # per-paddle plot
    # -------------------------------------------------
    xfit = np.linspace(np.nanmin(alt_roi), np.nanmax(alt_roi), 200)

    plt.figure(figsize=(7, 5))

    plt.plot(
        alt_roi,
        rate_roi,
        ".",
        alpha=0.35,
        label=f"{threshold_name} data"
    )

    plt.plot(
        xfit,
        line(xfit, m_fit, b_fit),
        "r-",
        lw=2,
        label=f"slope = {m_fit:.3e} ± {m_err:.3e} rate/m"
    )

    plt.xlabel("Altitude [m]")
    plt.ylabel(f"{threshold_name} rate")
    plt.title(
        f"Paddle {paddle_id}: {threshold_name} rate vs altitude\n"
        f"PA A: {tempA_mean-temp_half_window:.2f}–{tempA_mean+temp_half_window:.2f} C, "
        f"PA B: {tempB_mean-temp_half_window:.2f}–{tempB_mean+temp_half_window:.2f} C"
    )

    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    n_plots += 1


fit_df_alt_at_pa_temp = pd.DataFrame(fit_rows_alt_at_pa_temp)
fit_df_alt_at_pa_temp






df = fit_df_alt_at_pa_temp.copy()

df_low = df[df["paddle_id"] < 13].sort_values("paddle_id")
df_high = df[df["paddle_id"] >= 13].sort_values("paddle_id")


def make_plot_alt(df_sub, title_suffix):

    if len(df_sub) == 0:
        print(f"No data for {title_suffix}")
        return

    paddles_x = df_sub["paddle_id"].to_numpy()
    slopes = df_sub["rate_vs_alt_slope"].to_numpy()
    errors = df_sub["rate_vs_alt_slope_err"].to_numpy()
    npts = df_sub["n_points"].to_numpy()

    plt.figure(figsize=(7, 5))

    plt.errorbar(
        paddles_x,
        slopes,
        yerr=errors,
        fmt="o",
        capsize=4
    )

    for x, y, n in zip(paddles_x, slopes, npts):
        plt.text(x, y, f"{n}", fontsize=8, ha="center", va="bottom")

    good = np.isfinite(slopes) & np.isfinite(errors) & (errors > 0)

    if np.sum(good) > 0:
        weights = 1 / errors[good]**2
        wmean = np.sum(weights * slopes[good]) / np.sum(weights)
        werr = np.sqrt(1 / np.sum(weights))
        plt.axhline(wmean, color="r", lw=2, label=f"mean = {wmean:.3e} ± {werr:.3e}")

    plt.axhline(0, color="k", ls="--", alpha=0.5)

    plt.xlabel("Paddle ID")
    plt.ylabel("Slope [rate / m]")
    plt.title(
        f"{threshold_name} rate vs altitude slope ({title_suffix})\n"
        f"Each PA constrained to its own mean ± {temp_half_window:.1f} C"
    )

    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


make_plot_alt(df_low, "paddle_id < 13")
make_plot_alt(df_high, "paddle_id ≥ 13")